# Plots and Statistics for use in the Report

In [31]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]= '0'

from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import ast

In [32]:
# Load data
df_seen = pd.read_csv("data/results/wandb_results_seen.csv")
df_unseen_reflow = pd.read_csv("data/results/wandb_unseen_and_reflow.csv")
df_baselines = pd.read_csv("data/baseline_results.csv")

# Rename baseline columns for consistency
df_baselines.rename(columns={"DE_F1": "de_test_score", "DE_Precision": "de_precision_test_score", "DE_Recall": "de_recall_test_score", "Energy_Loss": "energy_test_score", "MSE": "mse_test_loss"}, inplace=True) 

In [33]:
df_baselines.columns

Index(['seed', 'task', 'baseline', 'de_test_score', 'de_precision_test_score',
       'de_recall_test_score', 'energy_test_score', 'mse_test_loss',
       'DE-F1_all', 'DE-Precision_all', 'DE-Recall_all', 'Energy_Loss_all',
       'MSE_all'],
      dtype='object')

In [34]:
df_unseen_reflow.columns

Index(['Name', 'State', 'Notes', 'User', 'Tags', 'Created', 'Runtime', 'Sweep',
       'HVG_GENE_NUMBER', 'architecture', 'batch_size', 'dataset', 'epochs',
       'learning_rate', 'loss_function', 'num_heads', 'num_layers',
       'MLP_hidden_size', 'dropout', 'hidden_size', 'n_context', 'n_steps',
       'perturbations', 'reflow', 'seed', 'weight_decay', 'embedding_dim',
       'gpu', 'model_family', 'parameters.batch_sizes',
       'parameters.embedding_dims', 'parameters.hidden_sizes',
       'parameters.num_layerss', 'early_stopping_patience', 'fingerprint_bits',
       'fingerprint_radius', 'gate_loss_weight',
       'parameters.gate_loss_weight.distribution',
       'parameters.gate_loss_weight.max', 'parameters.gate_loss_weight.min',
       'task', 'unseen_perturbations', 'epoch', 'test_loss', 'train_loss',
       'val_loss', 'de_val_score', 'de_precision_val_score',
       'de_recall_val_score', 'energy_val_score', 'mse_val_loss',
       'de_test_score', 'de_precision_test_sco

In [35]:
cols_of_interest = ["Name", "seed", "model_family", "reflow"]
val_metric_cols = ["de_val_score", "de_precision_val_score", "de_recall_val_score", "energy_val_score", "mse_val_loss"]
test_metric_cols = ["de_test_score", "de_precision_test_score", "de_recall_test_score", "energy_test_score", "mse_test_loss"]
mean_baseline_cols = ["mean_baseline_de_score", "mean_baseline_precision_score", "mean_baseline_recall_score", "mean_baseline_energy_score", "mean_baseline_mse_score"]
mean_shift_baseline_cols = ["mean_shift_baseline_de_score", "mean_shift_baseline_precision_score", "mean_shift_baseline_recall_score", "mean_shift_baseline_energy_score", "mean_shift_baseline_mse_score"]

all_values_cols = ["de_test_score_all", "de_precition_test_score_all", "de_recall_test_score_all", "energy_test_score_all", "mse_test_loss_all"]
all_baseline_cols = ["mean_baseline_de_score_all", "mean_baseline_precision_score_all", "mean_baseline_recall_score_all", "mean_baseline_energy_score_all", "mean_baseline_mse_score_all"]

perturbations = ["BI-3406", "Capmatinib", "Paclitaxel", "Oxaliplatin", "Gemcitabine", "Vemurafenib", "Anastrozole", "Minodronic acid", "Everolimus", "Adagrasib"]

num_degs = {
    "BI-3406": 33,
    "Capmatinib": 12,
    "Paclitaxel": 11,
    "Oxaliplatin": 8,
    "Gemcitabine": 8,
    "Vemurafenib": 6,
    "Anastrozole": 9,
    "Minodronic acid": 6,
    "Everolimus": 3,
    "Adagrasib": 1
}

num_degs_unseen = {
    "RMC-6236": 23,
    "4EGI-1": 10, 
    "AT7519": 5
}

# Used the one calculated on PCA with pertpy
energy_loss_by_pert = {
    "BI-3406": np.round(0.805105, 4),
    "Capmatinib": np.round(0.361640, 4),
    "Paclitaxel": np.round(0.342155, 4),
    "Oxaliplatin": np.round(0.221031, 4),
    "Gemcitabine": np.round(0.220253, 4),
    "Vemurafenib": np.round(0.202073, 4),
    "Anastrozole": np.round(0.192639, 4),
    "Minodronic acid": np.round(0.180063, 4),
    "Everolimus": np.round(0.163651, 4),
    "Adagrasib": np.round(0.113534, 4)
}

energy_loss_by_pert_unseen = {
    "RMC-6236": np.round(0.613523, 4),
    "4EGI-1": np.round(0.214009, 4), 
    "AT7519": np.round(0.147334, 4)
}

mse_loss_by_pert = {
    "BI-3406": np.round(0.185530, 4),
    "Capmatinib": np.round(0.079464, 4),
    "Paclitaxel": np.round(0.077663, 4),
    "Oxaliplatin": np.round(0.045760, 4),
    "Gemcitabine": np.round(0.044346, 4),
    "Vemurafenib": np.round(0.042177, 4),
    "Anastrozole": np.round(0.040109, 4),
    "Minodronic acid": np.round(0.035912, 4),
    "Everolimus": np.round(0.034036, 4),
    "Adagrasib":np.round( 0.021393, 4)
}

mse_loss_by_pert_unseen = {
    "RMC-6236": np.round(0.141892, 4),
    "4EGI-1": np.round(0.044071, 4), 
    "AT7519": np.round(0.029690, 4)
}

# Seen Evaluation

In [36]:
df_seen = df_seen[cols_of_interest + val_metric_cols + test_metric_cols + all_values_cols]
df_seen.head()

,Name,seed,model_family,reflow,de_val_score,de_precision_val_score,de_recall_val_score,energy_val_score,mse_val_loss,de_test_score,de_precision_test_score,de_recall_test_score,energy_test_score,mse_test_loss,de_test_score_all,de_precition_test_score_all,de_recall_test_score_all,energy_test_score_all,mse_test_loss_all
0,TRAINSEEN-RectifiedFlow-seed585960-true-jazz-3709,585960,RectifiedFlow,0,0.508680,0.707959,0.470047,0.047452,0.331921,0.532219,0.658401,0.474794,0.045753,0.330823,"[0.6067415730337079,0.5652173913043478,0.53658...","[0.6923076923076923,0.5909090909090909,0.61111...","[0.54,0.5416666666666666,0.4782608695652174,0....","[0.09177017211914062,0.04525184631347656,0.047...","[0.32390329241752625,0.31926465034484863,0.336..."
1,TRAINSEEN-RectifiedFlow-seed555657-silver-butt...,555657,RectifiedFlow,0,0.487390,0.611712,0.454589,0.049641,0.330563,0.554347,0.718149,0.483478,0.049331,0.329775,"[0.6336633663366336,0.6842105263157896,0.47619...","[0.6808510638297872,0.8125,0.7142857142857143,...","[0.5925925925925926,0.5909090909090909,0.35714...","[0.08614540100097656,0.05130767822265625,0.048...","[0.32353758811950684,0.31601399183273315,0.338..."
2,TRAINSEEN-RectifiedFlow-seed525354-divine-musi...,525354,RectifiedFlow,0,0.501198,0.718724,0.427803,0.048134,0.329682,0.524730,0.714323,0.477555,0.046601,0.332629,"[0.6458333333333334,0.7199999999999999,0.58064...","[0.7380952380952381,0.75,0.6428571428571429,0....","[0.5740740740740741,0.6923076923076923,0.52941...","[0.08834266662597656,0.04587554931640625,0.047...","[0.32340550422668457,0.318693608045578,0.34153..."
3,TRAINSEEN-Transformer-seed585960-fresh-thunder...,585960,Transformer,0,0.613902,0.680025,0.590678,0.039313,0.344974,0.672736,0.795516,0.600534,0.039847,0.339003,"[0.7294117647058823,0.717948717948718,0.681818...","[0.8857142857142857,0.7777777777777778,0.75,0....","[0.62,0.6666666666666666,0.625,0.3103448275862...","[0.036418914794921875,0.03859138488769531,0.03...","[0.34797191619873047,0.32623013854026794,0.348..."
4,TRAINSEEN-Transformer-seed555657-toasty-cloud-...,555657,Transformer,0,0.594990,0.769604,0.492891,0.038632,0.337871,0.583318,0.831371,0.457031,0.038650,0.347509,"[0.6666666666666665,0.6666666666666666,0.57894...","[0.8571428571428571,0.8181818181818182,0.73333...","[0.5454545454545454,0.5625,0.4782608695652174,...","[0.03686332702636719,0.03724861145019531,0.037...","[0.3534870743751526,0.33195751905441284,0.3578..."


In [37]:
ast.literal_eval(df_unseen_reflow["de_test_score_all"][0])

[0.5564516129032258, 0.1285714285714286, 0.2926829268292683]

In [38]:
df_rf_seen = df_seen.where(df_seen["model_family"] == "RectifiedFlow").dropna().reset_index(drop=True)
df_tf_seen = df_seen.where(df_seen["model_family"] == "TransformerFlow").dropna().reset_index(drop=True)
df_transformer_seen = df_seen.where(df_seen["model_family"] == "Transformer").dropna().reset_index(drop=True)   

In [39]:
df_baselines_seen = df_baselines.where(df_baselines["task"] == "SEEN").dropna().reset_index(drop=True)
df_mean_baseline = df_baselines_seen.where(df_baselines_seen["baseline"] == "mean").dropna().reset_index(drop=True)
df_mean_baseline_shift = df_baselines_seen.where(df_baselines_seen["baseline"] == "mean_shift").dropna().reset_index(drop=True)

In [40]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]= '0'

from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import ast
import re

In [41]:
# Load data
df_transformer = pd.read_csv("data/results/final/wandb_transformer_seen_unseen.csv")
df_rf = pd.read_csv("data/results/final/wandb_rectified_flow_seen_unseen.csv")
df_tf = pd.read_csv("data/results/final/wandb_transformerflow_seen_unseen.csv")
df_baselines = pd.read_csv("data/results/final/baseline_results.csv")
df_baselines_unseen_subsample = pd.read_csv("data/results/final/baseline_results_unseen_subsample.csv")

In [42]:
perturbations = ["BI-3406", "Capmatinib", "Paclitaxel", "Oxaliplatin", "Gemcitabine", "Vemurafenib", "Anastrozole", "Minodronic acid", "Everolimus", "Adagrasib"]
unseen_perturbations = ["RMC-6236", "4EGI-1", "AT7519"]

combined_perturbations = perturbations + unseen_perturbations

In [43]:
cols_of_interest = ["Name", "seed", "model_family", "task"]
val_metric_cols = ["de_val_score", "de_precision_val_score", "de_recall_val_score", "energy_val_score", "mse_val_loss"]
test_metric_cols = ["de_test_score", "de_precision_test_score", "de_recall_test_score", "energy_test_score", "mse_test_loss"]

baseline_cols_of_interest = ["task", "baseline", "seed"]
baseline_cols = ["DE_F1", "DE_Precision", "DE_Recall", "Energy_Loss", "MSE"]
baseline_cols_all = ["DE-F1_all", "DE-Precision_all", "DE-Recall_all", "Energy_Loss_all", "MSE_all"]

# mean_baseline_cols = ["mean_baseline_de_score", "mean_baseline_precision_score", "mean_baseline_recall_score", "mean_baseline_energy_score", "mean_baseline_mse_score"]
# mean_shift_baseline_cols = ["mean_shift_baseline_de_score", "mean_shift_baseline_precision_score", "mean_shift_baseline_recall_score", "mean_shift_baseline_energy_score", "mean_shift_baseline_mse_score"]

all_values_cols = ["de_test_score_all", "de_precition_test_score_all", "de_recall_test_score_all", "energy_test_score_all", "mse_test_loss_all"]

# cols_of_interest = ["Name", "seed", "model_family", "reflow"]
# val_metric_cols = ["de_val_score", "de_precision_val_score", "de_recall_val_score", "energy_val_score", "mse_val_loss"]
# test_metric_cols = ["de_test_score", "de_precision_test_score", "de_recall_test_score", "energy_test_score", "mse_test_loss"]
# mean_baseline_cols = ["mean_baseline_de_score", "mean_baseline_precision_score", "mean_baseline_recall_score", "mean_baseline_energy_score", "mean_baseline_mse_score"]
# mean_shift_baseline_cols = ["mean_shift_baseline_de_score", "mean_shift_baseline_precision_score", "mean_shift_baseline_recall_score", "mean_shift_baseline_energy_score", "mean_shift_baseline_mse_score"]

# all_values_cols = ["de_test_score_all", "de_precition_test_score_all", "de_recall_test_score_all", "energy_test_score_all", "mse_test_loss_all"]
# all_baseline_cols = ["mean_baseline_de_score_all", "mean_baseline_precision_score_all", "mean_baseline_recall_score_all", "mean_baseline_energy_score_all", "mean_baseline_mse_score_all"]

# perturbations = ["BI-3406", "Capmatinib", "Paclitaxel", "Oxaliplatin", "Gemcitabine", "Vemurafenib", "Anastrozole", "Minodronic acid", "Everolimus", "Adagrasib"]

num_degs = {
    "BI-3406": 33,
    "Capmatinib": 12,
    "Paclitaxel": 11,
    "Oxaliplatin": 8,
    "Gemcitabine": 8,
    "Vemurafenib": 6,
    "Anastrozole": 9,
    "Minodronic acid": 6,
    "Everolimus": 3,
    "Adagrasib": 1
}

num_degs_unseen = {
    "RMC-6236": 23,
    "4EGI-1": 10, 
    "AT7519": 5
}

# Used the one calculated on PCA with pertpy
energy_loss_by_pert = {
    "BI-3406": np.round(0.805105, 4),
    "Capmatinib": np.round(0.361640, 4),
    "Paclitaxel": np.round(0.342155, 4),
    "Oxaliplatin": np.round(0.221031, 4),
    "Gemcitabine": np.round(0.220253, 4),
    "Vemurafenib": np.round(0.202073, 4),
    "Anastrozole": np.round(0.192639, 4),
    "Minodronic acid": np.round(0.180063, 4),
    "Everolimus": np.round(0.163651, 4),
    "Adagrasib": np.round(0.113534, 4)
}

energy_loss_by_pert_unseen = {
    "RMC-6236": np.round(0.613523, 4),
    "4EGI-1": np.round(0.214009, 4), 
    "AT7519": np.round(0.147334, 4)
}

mse_loss_by_pert = {
    "BI-3406": np.round(0.185530, 4),
    "Capmatinib": np.round(0.079464, 4),
    "Paclitaxel": np.round(0.077663, 4),
    "Oxaliplatin": np.round(0.045760, 4),
    "Gemcitabine": np.round(0.044346, 4),
    "Vemurafenib": np.round(0.042177, 4),
    "Anastrozole": np.round(0.040109, 4),
    "Minodronic acid": np.round(0.035912, 4),
    "Everolimus": np.round(0.034036, 4),
    "Adagrasib":np.round( 0.021393, 4)
}

mse_loss_by_pert_unseen = {
    "RMC-6236": np.round(0.141892, 4),
    "4EGI-1": np.round(0.044071, 4), 
    "AT7519": np.round(0.029690, 4)
}

In [44]:
df_transformer = df_transformer[cols_of_interest + val_metric_cols + test_metric_cols + all_values_cols]
df_rf = df_rf[cols_of_interest + val_metric_cols + test_metric_cols + all_values_cols]
df_tf = df_tf[cols_of_interest + val_metric_cols + test_metric_cols + all_values_cols]
df_baselines = df_baselines[baseline_cols_of_interest + baseline_cols + baseline_cols_all]
df_baselines.rename(columns={"DE_F1": "de_test_score", "DE_Precision": "de_precision_test_score", "DE_Recall": "de_recall_test_score", "Energy_Loss": "energy_test_score", "MSE": "mse_test_loss"}, inplace=True) 

df_baselines_unseen_subsample = df_baselines_unseen_subsample[baseline_cols_of_interest + baseline_cols + baseline_cols_all]
df_baselines_unseen_subsample.rename(columns={"DE_F1": "de_test_score", "DE_Precision": "de_precision_test_score", "DE_Recall": "de_recall_test_score", "Energy_Loss": "energy_test_score", "MSE": "mse_test_loss"}, inplace=True)

In [45]:
df_mean_baseline = df_baselines[df_baselines["baseline"] == "mean"]
df_mean_shift_baseline = df_baselines[df_baselines["baseline"] == "mean_shift"]

df_mean_baseline_unseen_subsample = df_baselines_unseen_subsample[df_baselines_unseen_subsample["baseline"] == "mean"]
df_mean_shift_baseline_unseen_subsample = df_baselines_unseen_subsample[df_baselines_unseen_subsample["baseline"] == "mean_shift"]

In [46]:
df_transformer_seen = df_transformer[df_transformer["task"] == "SEEN"]
df_transformer_unseen = df_transformer[df_transformer["task"] == "UNSEEN-PERT"]

df_rf_seen = df_rf[df_rf["task"] == "SEEN"]
df_rf_unseen = df_rf[df_rf["task"] == "UNSEEN-PERT"]

df_tf_seen = df_tf[df_tf["task"] == "SEEN"]
df_tf_unseen = df_tf[df_tf["task"] == "UNSEEN-PERT"]

df_mean_baseline_seen = df_mean_baseline[df_mean_baseline["task"] == "SEEN"]
df_mean_baseline_unseen = df_mean_baseline[df_mean_baseline["task"] == "UNSEEN-PERT"]

df_mean_shift_baseline_seen = df_mean_shift_baseline[df_mean_shift_baseline["task"] == "SEEN"]
df_mean_shift_baseline_unseen = df_mean_shift_baseline[df_mean_shift_baseline["task"] == "UNSEEN-PERT"]

In [47]:
print(df_transformer_seen["de_test_score"].mean(), df_transformer_seen["de_test_score"].std())
print(df_rf_seen["de_test_score"].mean(), df_rf_seen["de_test_score"].std())
print(df_tf_seen["de_test_score"].mean(), df_tf_seen["de_test_score"].std())

0.6097015170509297 0.03734542724181878
0.5090797116362419 0.04116768412463953
0.46337753029921636 0.04323417503614575


In [48]:
np.round(df_transformer_seen.describe(), 4)

,seed,de_val_score,de_precision_val_score,de_recall_val_score,energy_val_score,mse_val_loss,de_test_score,de_precision_test_score,de_recall_test_score,energy_test_score,mse_test_loss
count,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000
mean,292074.0000,0.6105,0.7564,0.5423,0.0371,0.3425,0.6097,0.8013,0.5161,0.0386,0.3436
std,188456.3373,0.0270,0.0564,0.0502,0.0013,0.0025,0.0373,0.0523,0.0468,0.0017,0.0041
min,123.0000,0.5577,0.6685,0.4656,0.0345,0.3379,0.5094,0.7245,0.3862,0.0358,0.3386
25%,154142.2500,0.5937,0.7102,0.5007,0.0362,0.3414,0.5905,0.7694,0.4880,0.0374,0.3397
50%,298081.5000,0.6104,0.7381,0.5243,0.0367,0.3427,0.6219,0.7943,0.5226,0.0384,0.3435
75%,442020.7500,0.6349,0.8000,0.5913,0.0383,0.3445,0.6278,0.8324,0.5387,0.0392,0.3464
max,585960.0000,0.6620,0.8617,0.6418,0.0393,0.3463,0.6727,0.9313,0.6005,0.0426,0.3512


In [49]:
np.round(df_rf_seen.describe(), 4)

,seed,de_val_score,de_precision_val_score,de_recall_val_score,energy_val_score,mse_val_loss,de_test_score,de_precision_test_score,de_recall_test_score,energy_test_score,mse_test_loss
count,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000
mean,292074.0000,0.4633,0.7535,0.3689,0.0495,0.3346,0.5091,0.6930,0.4440,0.0543,0.3294
std,188456.3373,0.0392,0.1142,0.0508,0.0036,0.0032,0.0412,0.0898,0.0522,0.0079,0.0055
min,123.0000,0.3975,0.5341,0.2905,0.0442,0.3298,0.4214,0.5673,0.3446,0.0443,0.3169
25%,154142.2500,0.4323,0.6685,0.3303,0.0469,0.3317,0.4835,0.6425,0.4142,0.0485,0.3258
50%,298081.5000,0.4636,0.7647,0.3782,0.0490,0.3347,0.5077,0.6793,0.4376,0.0525,0.3289
75%,442020.7500,0.4930,0.8128,0.4010,0.0505,0.3373,0.5459,0.7525,0.4831,0.0572,0.3334
max,585960.0000,0.5316,0.9407,0.4814,0.0579,0.3396,0.5788,0.8848,0.5301,0.0745,0.3383


In [50]:
np.round(df_tf_seen.describe(), 4)

,seed,de_val_score,de_precision_val_score,de_recall_val_score,energy_val_score,mse_val_loss,de_test_score,de_precision_test_score,de_recall_test_score,energy_test_score,mse_test_loss
count,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000
mean,292074.0000,0.4282,0.6608,0.3518,0.0538,0.3304,0.4634,0.6218,0.4061,0.0541,0.3314
std,188456.3373,0.0389,0.0683,0.0403,0.0033,0.0028,0.0432,0.0712,0.0498,0.0036,0.0032
min,123.0000,0.3508,0.5547,0.2831,0.0488,0.3245,0.3994,0.4895,0.2938,0.0490,0.3257
25%,154142.2500,0.4066,0.6095,0.3291,0.0511,0.3284,0.4295,0.5494,0.3624,0.0507,0.3294
50%,298081.5000,0.4213,0.6640,0.3481,0.0544,0.3305,0.4625,0.6426,0.4177,0.0540,0.3314
75%,442020.7500,0.4486,0.6993,0.3703,0.0562,0.3326,0.4960,0.6653,0.4384,0.0562,0.3329
max,585960.0000,0.4969,0.8095,0.4410,0.0596,0.3343,0.5383,0.7500,0.4799,0.0627,0.3397


## Seen Plots

### Bar chart by model and perturbation separately

In [51]:
df_rf_seen.head()

,Name,seed,model_family,task,de_val_score,de_precision_val_score,de_recall_val_score,energy_val_score,mse_val_loss,de_test_score,de_precision_test_score,de_recall_test_score,energy_test_score,mse_test_loss,de_test_score_all,de_precition_test_score_all,de_recall_test_score_all,energy_test_score_all,mse_test_loss_all
4,TRAINSEEN-RectifiedFlow-seed585960-dashing-vio...,585960,RectifiedFlow,SEEN,0.505763,0.623695,0.481375,0.053400,0.331681,0.484497,0.656190,0.422894,0.051948,0.325900,"[0.5405405405405406,0.5555555555555556,0.37837...","[0.8333333333333334,0.8333333333333334,0.5,1,0...","[0.4,0.4166666666666667,0.30434782608695654,0....","[0.09473800659179688,0.0496063232421875,0.0597...","[0.32010263204574585,0.3145514130592346,0.3302..."
6,TRAINSEEN-RectifiedFlow-seed555657-elated-feat...,555657,RectifiedFlow,SEEN,0.418894,0.602492,0.392625,0.056372,0.337362,0.499262,0.750973,0.397274,0.056372,0.325779,"[0.5205479452054794,0.5161290322580644,0.41025...","[1,0.8888888888888888,0.7272727272727273,0.666...","[0.35185185185185186,0.3636363636363637,0.2857...","[0.09033966064453124,0.06025886535644531,0.051...","[0.32773473858833313,0.31338194012641907,0.334..."
7,TRAINSEEN-RectifiedFlow-seed525354-electric-wa...,525354,RectifiedFlow,SEEN,0.515391,0.801915,0.436783,0.046484,0.337284,0.532165,0.772294,0.447185,0.045454,0.336938,"[0.5853658536585367,0.6956521739130435,0.61538...","[0.8571428571428571,0.8,0.8888888888888888,0.7...","[0.4444444444444444,0.6153846153846154,0.47058...","[0.0715484619140625,0.048473358154296875,0.045...","[0.33083194494247437,0.3215508460998535,0.3433..."
9,TRAINSEEN-RectifiedFlow-seed495051-firm-sunset...,495051,RectifiedFlow,SEEN,0.421859,0.937381,0.292618,0.049879,0.331174,0.485236,0.651720,0.464524,0.050865,0.328868,"[0.45,0.5714285714285714,0.5142857142857142,0....","[0.9473684210526316,0.625,0.6923076923076923,0...","[0.29508196721311475,0.5263157894736842,0.4090...","[0.07776451110839844,0.05328178405761719,0.051...","[0.3293033540248871,0.31638702750205994,0.3343..."
11,TRAINSEEN-RectifiedFlow-seed464748-royal-firef...,464748,RectifiedFlow,SEEN,0.490338,0.778436,0.374093,0.050365,0.339590,0.548131,0.639704,0.530144,0.055104,0.334255,"[0.48837209302325585,0.625,0.6666666666666666,...","[0.8076923076923077,0.7692307692307693,0.66666...","[0.35,0.5263157894736842,0.6666666666666666,0....","[0.08702850341796875,0.05209541320800781,0.045...","[0.33433446288108826,0.3290834426879883,0.3439..."


### Combined DEGs X performance plot

In [52]:
num_degs_comb = {**num_degs, **num_degs_unseen}
df_num_degs_comb = pd.DataFrame.from_dict(num_degs_comb, orient='index', columns=['num_degs']).sort_values(by="num_degs", ascending=False)

edist_comb = {**energy_loss_by_pert, **energy_loss_by_pert_unseen}
df_edist_comb = pd.DataFrame.from_dict(edist_comb, orient='index', columns=['energy_loss']).sort_values(by="energy_loss", ascending=False)
df_edist_comb

,energy_loss
BI-3406,0.8051
RMC-6236,0.6135
Capmatinib,0.3616
Paclitaxel,0.3422
Oxaliplatin,0.2210
Gemcitabine,0.2203
4EGI-1,0.2140
Vemurafenib,0.2021
Anastrozole,0.1926
Minodronic acid,0.1801


In [53]:
perturbations

['BI-3406',
 'Capmatinib',
 'Paclitaxel',
 'Oxaliplatin',
 'Gemcitabine',
 'Vemurafenib',
 'Anastrozole',
 'Minodronic acid',
 'Everolimus',
 'Adagrasib']

In [54]:
# Mean Baseline
mean_baseline_de_by_pert = []
for idx, row in df_mean_baseline_seen.iterrows():
    mean_baseline_de_by_pert.append(ast.literal_eval(row["DE-F1_all"]))
mean_baseline_de_by_pert = np.array(mean_baseline_de_by_pert)

mean_baseline_de_by_pert_unseen = []
for idx, row in df_mean_baseline_unseen_subsample.iterrows():
    mean_baseline_de_by_pert_unseen.append(ast.literal_eval(row["DE-F1_all"]))
mean_baseline_de_by_pert_unseen = np.array(mean_baseline_de_by_pert_unseen)
###
df_mean_baseline_de_by_pert_seen = pd.DataFrame(mean_baseline_de_by_pert, columns=perturbations)
df_mean_baseline_de_by_pert_unseen = pd.DataFrame(mean_baseline_de_by_pert_unseen, columns=unseen_perturbations)

# Comb
df_mean_baseline_de_by_pert_comb = pd.concat([df_mean_baseline_de_by_pert_seen, df_mean_baseline_de_by_pert_unseen], axis=1)
df_mean_baseline_de_by_pert_comb_mean = df_mean_baseline_de_by_pert_comb.mean(axis=0)

df_mean_baseline_de_by_pert_comb_mean = pd.DataFrame(df_mean_baseline_de_by_pert_comb_mean, columns=["mean_DE-F1"], index=df_mean_baseline_de_by_pert_comb_mean.index)
df_mean_baseline_de_by_pert_comb_degs = pd.concat([df_mean_baseline_de_by_pert_comb_mean, df_num_degs_comb, df_edist_comb], axis=1).sort_values(by="num_degs", ascending=False)


# mean_baseline_de_by_pert_comb = np.concatenate((mean_baseline_de_by_pert, mean_baseline_de_by_pert_unseen), axis=1)


# Mean Shift Baseline
mean_shift_baseline_de_by_pert = []
for idx, row in df_mean_shift_baseline_seen.iterrows():
    mean_shift_baseline_de_by_pert.append(ast.literal_eval(row["DE-F1_all"]))
mean_shift_baseline_de_by_pert = np.array(mean_shift_baseline_de_by_pert)

mean_shift_baseline_de_by_pert_unseen = []
for idx, row in df_mean_shift_baseline_unseen_subsample.iterrows():
    mean_shift_baseline_de_by_pert_unseen.append(ast.literal_eval(row["DE-F1_all"]))
mean_shift_baseline_de_by_pert_unseen = np.array(mean_shift_baseline_de_by_pert_unseen)

df_mean_shift_baseline_de_by_pert_seen = pd.DataFrame(mean_shift_baseline_de_by_pert, columns=perturbations)
df_mean_shift_baseline_de_by_pert_unseen = pd.DataFrame(mean_shift_baseline_de_by_pert_unseen, columns=unseen_perturbations)

df_mean_shift_baseline_de_by_pert_comb = pd.concat([df_mean_shift_baseline_de_by_pert_seen, df_mean_shift_baseline_de_by_pert_unseen], axis=1)
df_mean_shift_baseline_de_by_pert_comb_mean = df_mean_shift_baseline_de_by_pert_comb.mean(axis=0)

df_mean_shift_baseline_de_by_pert_comb_mean = pd.DataFrame(df_mean_shift_baseline_de_by_pert_comb_mean, columns=["mean_DE-F1"], index=df_mean_shift_baseline_de_by_pert_comb_mean.index)
df_mean_shift_baseline_de_by_pert_comb_degs = pd.concat([df_mean_shift_baseline_de_by_pert_comb_mean, df_num_degs_comb, df_edist_comb], axis=1).sort_values(by="energy_loss", ascending=False)


# mean_shift_baseline_de_by_pert_comb = np.concatenate((mean_shift_baseline_de_by_pert, mean_shift_baseline_de_by_pert_unseen), axis=1)

df_mean_shift_baseline_de_by_pert_comb_degs.head(40)

,mean_DE-F1,num_degs,energy_loss
BI-3406,0.266317,33,0.8051
RMC-6236,0.130177,23,0.6135
Capmatinib,0.144569,12,0.3616
Paclitaxel,0.136885,11,0.3422
Oxaliplatin,0.144683,8,0.2210
Gemcitabine,0.109624,8,0.2203
4EGI-1,0.058269,10,0.2140
Vemurafenib,0.127381,6,0.2021
Anastrozole,0.079943,9,0.1926
Minodronic acid,0.082624,6,0.1801


In [55]:
num_degs

{'BI-3406': 33,
 'Capmatinib': 12,
 'Paclitaxel': 11,
 'Oxaliplatin': 8,
 'Gemcitabine': 8,
 'Vemurafenib': 6,
 'Anastrozole': 9,
 'Minodronic acid': 6,
 'Everolimus': 3,
 'Adagrasib': 1}

In [56]:
num_degs_unseen

{'RMC-6236': 23, '4EGI-1': 10, 'AT7519': 5}

In [57]:
num_degs_comb = {**num_degs, **num_degs_unseen}
df_num_degs_comb = pd.DataFrame.from_dict(num_degs_comb, orient='index', columns=['num_degs']).sort_values(by="num_degs", ascending=False)
df_num_degs_comb.to_dict()


{'num_degs': {'BI-3406': 33,
  'RMC-6236': 23,
  'Capmatinib': 12,
  'Paclitaxel': 11,
  '4EGI-1': 10,
  'Anastrozole': 9,
  'Oxaliplatin': 8,
  'Gemcitabine': 8,
  'Vemurafenib': 6,
  'Minodronic acid': 6,
  'AT7519': 5,
  'Everolimus': 3,
  'Adagrasib': 1}}

In [60]:
mean_baseline_de_by_pert_comb

NameError: name 'mean_baseline_de_by_pert_comb' is not defined

### DE-F1

In [59]:
# RF
rf_de_by_pert = []
for idx, row in df_rf_seen.iterrows():
    rf_de_by_pert.append(ast.literal_eval(row["de_test_score_all"]))
rf_de_by_pert = np.array(rf_de_by_pert)

# Transformer
transformer_de_by_pert = []
for idx, row in df_transformer_seen.iterrows():
    transformer_de_by_pert.append(ast.literal_eval(row["de_test_score_all"]))
transformer_de_by_pert = np.array(transformer_de_by_pert)

# TF
tf_de_by_pert = []
for idx, row in df_tf_seen.iterrows():
    tf_de_by_pert.append(ast.literal_eval(row["de_test_score_all"]))
tf_de_by_pert = np.array(tf_de_by_pert)

# Mean Baseline
mean_baseline_de_by_pert = []
for idx, row in df_mean_baseline_seen.iterrows():
    mean_baseline_de_by_pert.append(ast.literal_eval(row["DE-F1_all"]))
mean_baseline_de_by_pert = np.array(mean_baseline_de_by_pert)


# Mean Shift Baseline
mean_shift_baseline_de_by_pert = []
for idx, row in df_mean_shift_baseline_seen.iterrows():
    mean_shift_baseline_de_by_pert.append(ast.literal_eval(row["DE-F1_all"]))
mean_shift_baseline_de_by_pert = np.array(mean_shift_baseline_de_by_pert)


print(rf_de_by_pert.mean(axis=0))
print(transformer_de_by_pert.mean(axis=0))
print(tf_de_by_pert.mean(axis=0))

[0.49096667 0.56702083 0.55023516 0.34568395 0.63044897 0.43280878
 0.51283735 0.49135719 0.62494687 0.44449134]
[0.67775803 0.60062158 0.60180966 0.42326459 0.62912422 0.58510743
 0.64863461 0.66968836 0.63850308 0.62250361]
[0.5690246  0.53540798 0.50265912 0.31329367 0.60432638 0.33453682
 0.39854203 0.37526404 0.59427045 0.40645022]


In [61]:
mean_shift_baseline_de_by_pert.mean(axis=0)

array([0.26631711, 0.14456914, 0.13688516, 0.14468321, 0.10962352,
       0.12738078, 0.07994341, 0.08262436, 0.08338542, 0.05085863])

In [62]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

model_scores = {
    "Transformer": transformer_de_by_pert,
    "Rectified Flow": rf_de_by_pert,
    "TransformerFlow": tf_de_by_pert,
    "Mean Baseline": mean_baseline_de_by_pert,
    "Mean Shift Baseline": mean_shift_baseline_de_by_pert,
}
colors = px.colors.qualitative.Plotly
degs = pd.Series([num_degs[p] for p in perturbations], index=perturbations)
edistances = pd.Series([np.round(energy_loss_by_pert[p], 4).item() for p in perturbations], index=perturbations)


fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.75, 0.25], specs=[[{}], [{"secondary_y": True}]]) # 72 28

# Top: DE-F1 bars per model (legend shows Spearman rho between num DEGs and mean DE-F1)
for color, (model, scores) in zip(colors, model_scores.items()):
    means = pd.Series(scores.mean(axis=0), index=perturbations)
    # rho = means.corr(degs, method="spearman")
    rho = means.corr(edistances, method="spearman")
    # fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
    #                      name=f"{model} (ρ = {rho:.2f})", marker_color=color), row=1, col=1)
    print(f"{model}: Spearman ρ = {rho:.2f}")
    fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
                         name=f"{model}", marker_color=color), row=1, col=1)

# Bottom: perturbation magnitude (number of DEGs)
fig.add_trace(go.Scatter(x=perturbations, y=degs, mode="lines+markers+text", text=degs, textposition="bottom left",
                         line=dict(color="red"), name="# DEGs", showlegend=False), row=2, col=1, secondary_y=True)
fig.add_trace(go.Scatter(x=perturbations, y=edistances, mode="lines+markers+text", text=edistances, textposition="top center",
                         line=dict(color="black"), name="E-distance", showlegend=False), row=2, col=1, secondary_y=False)


# fig.update_layout(barmode="group", title="DE-F1 Score mean (↑) by Model and Perturbation vs. Number of DEGs",
#                   legend_title_text="Model (Spearman ρ with # DEGs)", font=dict(size=12))

fig.update_layout(barmode="group", title="DE-F1 Score mean (↑) by Model and Perturbation vs. E-distance of Perturbation",
                  legend_title_text="Model", font=dict(size=25))
# fig.update_layout(barmode="group", title="DE-F1 Score mean (↑) by Model and Perturbation vs. Number of DEGs",
#                   legend_title_text="Model", font=dict(size=25))

# fig.update_layout(legend=dict(
#     yanchor="top",
#     y=0.99,
#     xanchor="left",
#     x=0.01
# ))

fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.0,
    xanchor="left",
    x=0.01,
    itemsizing="constant",
))

fig.update_yaxes(title_text="DE-F1 Score", row=1, col=1)

fig.update_yaxes(title_text="# DEGs", range=[-10, 50], row=2, col=1, secondary_y=True, title_font_color="red") # * degs.max() * 1.5
fig.update_yaxes(title_text="E-distance", range=[-0.2, 1], row=2, col=1, secondary_y=False) # * degs.max() * 1.5
fig.update_xaxes(title_text="Perturbation", row=2, col=1)

# Set y-axis ticks for the second subplot
fig.update_yaxes(tickvals=[0, 10, 20, 30, 40, 50], row=2, col=1, secondary_y=True, tickfont_color="red")
fig.update_yaxes(tickvals=[0, 0.2, 0.4, 0.6, 0.8, 1], row=2, col=1, secondary_y=False)

# Tilt x-axis labels for better readability
fig.update_xaxes(tickangle=25, row=2, col=1)

fig.show()

fig.update_layout(
    font=dict(
        size=25
    )
)
# Save the figure as a PNG file
fig.write_image("thesis_plots/results/seen/de_f1_score_by_model_and_perturbation.png", scale=5, width=1400, height=1000)


Transformer: Spearman ρ = -0.15
Rectified Flow: Spearman ρ = 0.03
TransformerFlow: Spearman ρ = 0.16
Mean Baseline: Spearman ρ = 0.89
Mean Shift Baseline: Spearman ρ = 0.90


### DE-F1 Precision

In [63]:
# RF
rf_de_precision_by_pert = []
for idx, row in df_rf_seen.iterrows():
    rf_de_precision_by_pert.append(ast.literal_eval(row["de_precition_test_score_all"]))
rf_de_precision_by_pert = np.array(rf_de_precision_by_pert)

# Transformer
transformer_de_precision_by_pert = []
for idx, row in df_transformer_seen.iterrows():
    transformer_de_precision_by_pert.append(ast.literal_eval(row["de_precition_test_score_all"]))
transformer_de_precision_by_pert = np.array(transformer_de_precision_by_pert)

# TF
tf_de_precision_by_pert = []
for idx, row in df_tf_seen.iterrows():
    tf_de_precision_by_pert.append(ast.literal_eval(row["de_precition_test_score_all"]))
tf_de_precision_by_pert = np.array(tf_de_precision_by_pert)

# Mean Baseline
mean_baseline_de_precision_by_pert = []
for idx, row in df_mean_baseline_seen.iterrows():
    mean_baseline_de_precision_by_pert.append(ast.literal_eval(row["DE-Precision_all"]))
mean_baseline_de_precision_by_pert = np.array(mean_baseline_de_precision_by_pert)

# Mean Shift Baseline
mean_shift_baseline_de_precision_by_pert = []
for idx, row in df_mean_shift_baseline_seen.iterrows():
    mean_shift_baseline_de_precision_by_pert.append(ast.literal_eval(row["DE-Precision_all"]))
mean_shift_baseline_de_precision_by_pert = np.array(mean_shift_baseline_de_precision_by_pert)

In [64]:
# df_bar_des_precision_dict = {"Model": ["Transformer", "Rectified Flow", "TransformerFlow", "Mean Baseline", "Mean Shift Baseline"]}

# for i, pert in enumerate(perturbations):
#     df_bar_des_precision_dict[pert] = [transformer_de_precision_by_pert[:, i].mean().item(), rf_de_precision_by_pert[:, i].mean().item(), tf_de_precision_by_pert[:, i].mean().item(), mean_baseline_de_precision_by_pert[:, i].mean().item(), mean_shift_baseline_de_precision_by_pert[:, i].mean().item()]

# df_bar_des_precision = pd.DataFrame(df_bar_des_precision_dict)

# df_bar_des_precision_melted = df_bar_des_precision.melt(id_vars="Model", var_name="Perturbation", value_name="DE-Precision Score")
# fig = px.bar(df_bar_des_precision_melted, x="Perturbation", y="DE-Precision Score", color="Model", barmode="group", title="DE-Precision Score mean (↑) by Model and Perturbation")
# # error bars
# fig.update_traces(error_y=dict(type='data', array=[transformer_de_precision_by_pert[:, i].std() for i in range(len(perturbations))] + [rf_de_precision_by_pert[:, i].std() for i in range(len(perturbations))] + [tf_de_precision_by_pert[:, i].std() for i in range(len(perturbations))] + [mean_baseline_de_precision_by_pert[:, i].std() for i in range(len(perturbations))] + [mean_shift_baseline_de_precision_by_pert[:, i].std() for i in range(len(perturbations))], visible=True))
# fig.show()


In [65]:
print(rf_de_precision_by_pert.mean(axis=0))
print(transformer_de_precision_by_pert.mean(axis=0))
print(tf_de_precision_by_pert.mean(axis=0))
print(mean_baseline_de_precision_by_pert.mean(axis=0))
print(mean_shift_baseline_de_precision_by_pert.mean(axis=0))

[0.84759411 0.81268967 0.73745525 0.71316406 0.71429182 0.64316843
 0.70672619 0.70542582 0.60184154 0.44800325]
[0.88452937 0.76285778 0.73168652 0.87771825 0.83592075 0.75430529
 0.85564935 0.86736111 0.80197511 0.64059524]
[0.78669396 0.75506747 0.59046408 0.62380952 0.67400072 0.54206125
 0.67456349 0.57707071 0.57253942 0.4214881 ]
[0.12452981 0.0564449  0.05702265 0.05870921 0.04465461 0.05650625
 0.03330276 0.03120193 0.02946703 0.01512809]
[0.15406133 0.07818443 0.07375592 0.07848816 0.05825716 0.06855931
 0.04191378 0.04326404 0.04363708 0.02623339]


In [66]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

model_scores = {
    "Transformer": transformer_de_precision_by_pert,
    "Rectified Flow": rf_de_precision_by_pert,
    "TransformerFlow": tf_de_precision_by_pert,
    "Mean Baseline": mean_baseline_de_precision_by_pert,
    "Mean Shift Baseline": mean_shift_baseline_de_precision_by_pert,
}
colors = px.colors.qualitative.Plotly
degs = pd.Series([num_degs[p] for p in perturbations], index=perturbations)

fig = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[1.0]) # 72 28

# Top: DE-F1 bars per model (legend shows Spearman rho between num DEGs and mean DE-F1)
for color, (model, scores) in zip(colors, model_scores.items()):
    means = pd.Series(scores.mean(axis=0), index=perturbations)
    corr = means.corr(edistances, method="spearman")
    print(f"{model}: Spearman ρ = {corr:.2f}")
    fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
                         name=f"{model}", marker_color=color), row=1, col=1)

# Bottom: perturbation magnitude (number of DEGs)
# fig.add_trace(go.Scatter(x=perturbations, y=degs, mode="lines+markers+text", text=degs, textposition="top center",
#                          line=dict(color="black"), name="# DEGs", showlegend=False), row=1, col=1)

fig.update_layout(barmode="group", title="DE Precision Score mean (↑) by Model and Perturbation",
                  legend_title_text="Model", font=dict(size=25))

fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.0,
    xanchor="left",
    x=0.01
))

fig.update_yaxes(title_text="DE-Precision Score", row=1, col=1)
# fig.update_yaxes(title_text="# DEGs", range=[0, 50], row=2, col=1) # * degs.max() * 1.5
fig.update_xaxes(title_text="Perturbation", row=1, col=1)

# Tilt x-axis labels for better readability
fig.update_xaxes(tickangle=25, row=1, col=1)

# Save the figure as a PNG file
fig.write_image("thesis_plots/results/seen/de_precision_score_by_model_and_perturbation.png", scale=5, width=1400, height=700)

fig.show()

Transformer: Spearman ρ = 0.32
Rectified Flow: Spearman ρ = 0.95
TransformerFlow: Spearman ρ = 0.77
Mean Baseline: Spearman ρ = 0.89
Mean Shift Baseline: Spearman ρ = 0.90


### DE-F1 Recall

In [67]:
# RF
rf_de_recall_by_pert = []
for idx, row in df_rf_seen.iterrows():
    rf_de_recall_by_pert.append(ast.literal_eval(row["de_recall_test_score_all"]))
rf_de_recall_by_pert = np.array(rf_de_recall_by_pert)

# Transformer
transformer_de_recall_by_pert = []
for idx, row in df_transformer_seen.iterrows():
    transformer_de_recall_by_pert.append(ast.literal_eval(row["de_recall_test_score_all"]))
transformer_de_recall_by_pert = np.array(transformer_de_recall_by_pert)

# TF
tf_de_recall_by_pert = []
for idx, row in df_tf_seen.iterrows():
    tf_de_recall_by_pert.append(ast.literal_eval(row["de_recall_test_score_all"]))
tf_de_recall_by_pert = np.array(tf_de_recall_by_pert)

# Mean Baseline
mean_baseline_de_recall_by_pert = []
for idx, row in df_mean_baseline_seen.iterrows():
    mean_baseline_de_recall_by_pert.append(ast.literal_eval(row["DE-Recall_all"]))
mean_baseline_de_recall_by_pert = np.array(mean_baseline_de_recall_by_pert)

# Mean Shift Baseline
mean_shift_baseline_de_recall_by_pert = []
for idx, row in df_mean_shift_baseline_seen.iterrows():
    mean_shift_baseline_de_recall_by_pert.append(ast.literal_eval(row["DE-Recall_all"]))
mean_shift_baseline_de_recall_by_pert = np.array(mean_shift_baseline_de_recall_by_pert)

In [68]:
# df_bar_des_recall_dict = {"Model": ["Transformer", "Rectified Flow", "TransformerFlow", "Mean Baseline", "Mean Shift Baseline"]}

# for i, pert in enumerate(perturbations):
#     df_bar_des_recall_dict[pert] = [transformer_de_recall_by_pert[:, i].mean().item(), rf_de_recall_by_pert[:, i].mean().item(), tf_de_recall_by_pert[:, i].mean().item(), mean_baseline_de_recall_by_pert[:, i].mean().item(), mean_shift_baseline_de_recall_by_pert[:, i].mean().item()]

# df_bar_des_recall = pd.DataFrame(df_bar_des_recall_dict)

# df_bar_des_recall_melted = df_bar_des_recall.melt(id_vars="Model", var_name="Perturbation", value_name="DE-Recall Score")
# fig = px.bar(df_bar_des_recall_melted, x="Perturbation", y="DE-Recall Score", color="Model", barmode="group", title="DE-Recall Score mean (↑) by Model and Perturbation")
# # error bars
# fig.update_traces(error_y=dict(type='data', array=[transformer_de_recall_by_pert[:, i].std() for i in range(len(perturbations))] + [rf_de_recall_by_pert[:, i].std() for i in range(len(perturbations))] + [tf_de_recall_by_pert[:, i].std() for i in range(len(perturbations))] + [mean_baseline_de_recall_by_pert[:, i].std() for i in range(len(perturbations))] + [mean_shift_baseline_de_recall_by_pert[:, i].std() for i in range(len(perturbations))], visible=True))
# fig.show()

In [69]:
print(rf_de_recall_by_pert.mean(axis=0))
print(transformer_de_recall_by_pert.mean(axis=0))
print(tf_de_recall_by_pert.mean(axis=0))
print(mean_baseline_de_recall_by_pert.mean(axis=0))
print(mean_shift_baseline_de_recall_by_pert.mean(axis=0))

[0.35174147 0.44809512 0.45212978 0.23506681 0.58034627 0.34337768
 0.41360181 0.39122166 0.68630217 0.53833333]
[0.55260745 0.50496106 0.51760539 0.28360131 0.51751311 0.48448779
 0.53812815 0.56050827 0.55104829 0.65059524]
[0.45114784 0.42427316 0.44882197 0.21681873 0.55892599 0.25129478
 0.29940776 0.30171439 0.65136759 0.45714286]
[0.94336091 0.89111871 0.91048798 0.93087568 0.89657924 0.92007746
 0.93755514 0.96217426 0.84720415 0.88920635]
[0.99214875 0.99045214 0.97487179 0.95938353 0.96655899 0.9216243
 0.91361615 0.97189649 0.97686772 0.94396825]


In [70]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

model_scores = {
    "Transformer": transformer_de_recall_by_pert,
    "Rectified Flow": rf_de_recall_by_pert,
    "TransformerFlow": tf_de_recall_by_pert,
    "Mean Baseline": mean_baseline_de_recall_by_pert,
    "Mean Shift Baseline": mean_shift_baseline_de_recall_by_pert,
}
colors = px.colors.qualitative.Plotly
degs = pd.Series([num_degs[p] for p in perturbations], index=perturbations)

fig = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[1.0]) # 72 28

# Top: DE-F1 bars per model (legend shows Spearman rho between num DEGs and mean DE-F1)
for color, (model, scores) in zip(colors, model_scores.items()):
    means = pd.Series(scores.mean(axis=0), index=perturbations)
    corr = means.corr(edistances, method="spearman")
    print(f"{model}: Spearman ρ = {corr:.2f}")
    fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
                         name=f"{model}", marker_color=color), row=1, col=1)

# Bottom: perturbation magnitude (number of DEGs)
# fig.add_trace(go.Scatter(x=perturbations, y=degs, mode="lines+markers+text", text=degs, textposition="top center",
#                          line=dict(color="black"), name="# DEGs", showlegend=False), row=1, col=1)

fig.update_layout(barmode="group", title="DE Recall Score mean (↑) by Model and Perturbation",
                  legend_title_text="Model", font=dict(size=25))

fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.0,
    xanchor="left",
    x=0.01
))

fig.update_yaxes(title_text="DE-Recall Score", row=1, col=1)
# fig.update_yaxes(title_text="# DEGs", range=[0, 50], row=2, col=1) # * degs.max() * 1.5
fig.update_xaxes(title_text="Perturbation", row=1, col=1)

# Tilt x-axis labels for better readability
fig.update_xaxes(tickangle=25, row=1, col=1)

# Save the figure as a PNG file
fig.write_image("thesis_plots/results/seen/de_recall_score_by_model_and_perturbation.png", scale=5, width=1400, height=700)

fig.show()

Transformer: Spearman ρ = -0.48
Rectified Flow: Spearman ρ = -0.38
TransformerFlow: Spearman ρ = -0.20
Mean Baseline: Spearman ρ = 0.28
Mean Shift Baseline: Spearman ρ = 0.53


In [71]:
x1 = pd.Series([4.0, 4, 4, 4, 4, 4, 4, 4, 4, 4.00000001])
x2 = pd.Series([0.8, 0.6, 0.5, 0.4, 0.35, 0.3, 0.25, 0.2, 0.15, 0.1])
corr = x1.corr(x2, method="spearman")
corr

np.float64(-0.5222329678670935)

### Energy Score

In [72]:
# RF 
rf_energy_by_pert = []
for idx, row in df_rf_seen.iterrows():
    rf_energy_by_pert.append(ast.literal_eval(row["energy_test_score_all"]))
rf_energy_by_pert = np.array(rf_energy_by_pert)

# Transformer
transformer_energy_by_pert = []
for idx, row in df_transformer_seen.iterrows():
    transformer_energy_by_pert.append(ast.literal_eval(row["energy_test_score_all"]))
transformer_energy_by_pert = np.array(transformer_energy_by_pert)

# TF
tf_energy_by_pert = []
for idx, row in df_tf_seen.iterrows():
    tf_energy_by_pert.append(ast.literal_eval(row["energy_test_score_all"]))
tf_energy_by_pert = np.array(tf_energy_by_pert)

# Mean Baseline
mean_baseline_energy_by_pert = []
for idx, row in df_mean_baseline_seen.iterrows():
    mean_baseline_energy_by_pert.append(ast.literal_eval(row["Energy_Loss_all"]))
mean_baseline_energy_by_pert = np.array(mean_baseline_energy_by_pert)

# Mean Shift Baseline
mean_shift_baseline_energy_by_pert = []
for idx, row in df_mean_shift_baseline_seen.iterrows():
    mean_shift_baseline_energy_by_pert.append(ast.literal_eval(row["Energy_Loss_all"]))
mean_shift_baseline_energy_by_pert = np.array(mean_shift_baseline_energy_by_pert)

In [73]:
# df_bar_des_energy_dict = {"Model": ["Transformer", "Rectified Flow", "TransformerFlow", "Mean Baseline", "Mean Shift Baseline"]}

# for i, pert in enumerate(perturbations):
#     df_bar_des_energy_dict[pert] = [transformer_energy_by_pert[:, i].mean().item(), rf_energy_by_pert[:, i].mean().item(), tf_energy_by_pert[:, i].mean().item(), mean_baseline_energy_by_pert[:, i].mean().item(), mean_shift_baseline_energy_by_pert[:, i].mean().item()]

# df_bar_des_energy = pd.DataFrame(df_bar_des_energy_dict)

# df_bar_des_energy_melted = df_bar_des_energy.melt(id_vars="Model", var_name="Perturbation", value_name="Energy Score")
# fig = px.bar(df_bar_des_energy_melted, x="Perturbation", y="Energy Score", color="Model", barmode="group", log_y=True, title="Energy Score mean (↓) by Model and Perturbation")
# # error bars
# fig.update_traces(error_y=dict(type='data', array=[transformer_energy_by_pert[:, i].std() for i in range(len(perturbations))] + [rf_energy_by_pert[:, i].std() for i in range(len(perturbations))] + [tf_energy_by_pert[:, i].std() for i in range(len(perturbations))] + [mean_baseline_energy_by_pert[:, i].std() for i in range(len(perturbations))] + [mean_shift_baseline_energy_by_pert[:, i].std() for i in range(len(perturbations))], visible=True))
# fig.show()





from plotly.subplots import make_subplots
import plotly.graph_objects as go

model_scores = {
    "Transformer": transformer_energy_by_pert,
    "Rectified Flow": rf_energy_by_pert,
    "TransformerFlow": tf_energy_by_pert,
    "Mean Baseline": mean_baseline_energy_by_pert,
    "Mean Shift Baseline": mean_shift_baseline_energy_by_pert,
}
colors = px.colors.qualitative.Plotly
# degs = pd.Series([num_degs[p] for p in perturbations], index=perturbations)
edistances = pd.Series([np.round(energy_loss_by_pert[p], 4).item() for p in perturbations], index=perturbations)


fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.75, 0.25]) # 72 28

# Top: Energy bars per model
for color, (model, scores) in zip(colors, model_scores.items()):
    means = pd.Series(scores.mean(axis=0), index=perturbations)
    # rho = means.corr(degs, method="spearman")
    # fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
    #                      name=f"{model} (ρ = {rho:.2f})", marker_color=color), row=1, col=1)

    corr = means.corr(edistances, method="spearman")
    print(f"{model}: Spearman ρ = {corr:.2f}")

    fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
                         name=f"{model}", marker_color=color), row=1, col=1)

    # Set y-axis to log scale for energy loss
    fig.update_yaxes(type="log", row=1, col=1, tickvals=[0, 0.01, 0.1, 1, 5], ticktext=['0', '0.01', '0.1', '1', '5']) #tickvals=[0.01, 0.05, 0.1, 0.5, 1, 5], ticktext=['0.01', '0.05', '0.1', '0.5', '1', '5'])
     
        # Der Text, der an diesen Positionen angezeigt werden soll
        

# Bottom: perturbation magnitude (number of DEGs)
fig.add_trace(go.Scatter(x=perturbations, y=edistances, mode="lines+markers+text", text=edistances, textposition="top center",
                         line=dict(color="black"), name="E-Distance", showlegend=False), row=2, col=1)

# fig.update_layout(barmode="group", title="DE-F1 Score mean (↑) by Model and Perturbation vs. Number of DEGs",
#                   legend_title_text="Model (Spearman ρ with # DEGs)", font=dict(size=12))

fig.update_layout(barmode="group", title="E-Distance mean (↓) by Model and Perturbation vs. E-distance of Perturbation",
                  legend_title_text="Model", font=dict(size=25))

# fig.update_layout(legend=dict(
#     yanchor="top",
#     y=0.99,
#     xanchor="left",
#     x=0.01
# ))

fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.0,
    xanchor="left",
    x=0.01
))

fig.update_yaxes(title_text="E-Distance (log)", row=1, col=1)
fig.update_yaxes(title_text="E-Distance", range=[0, 1.2], row=2, col=1) # * degs.max() * 1.5
fig.update_xaxes(title_text="Perturbation", row=2, col=1)

# Tilt x-axis labels for better readability
fig.update_xaxes(tickangle=25, row=2, col=1)

# Save the figure as a PNG file
fig.write_image("thesis_plots/results/seen/edistance_by_model_and_perturbation.png", scale=5, width=1400, height=1000)

fig.show()

Transformer: Spearman ρ = 0.19
Rectified Flow: Spearman ρ = 0.83
TransformerFlow: Spearman ρ = 0.88
Mean Baseline: Spearman ρ = 0.05
Mean Shift Baseline: Spearman ρ = 0.72


In [74]:
# from plotly.subplots import make_subplots
# import plotly.graph_objects as go

# model_scores = {
#     "Transformer": transformer_de_by_pert,
#     "Rectified Flow": rf_de_by_pert,
#     "TransformerFlow": tf_de_by_pert,
#     "Mean Baseline": mean_baseline_de_by_pert,
#     "Mean Shift Baseline": mean_shift_baseline_de_by_pert,
# }
# colors = px.colors.qualitative.Plotly
# degs = pd.Series([num_degs[p] for p in perturbations], index=perturbations)

# fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.75, 0.25]) # 72 28

# # Top: DE-F1 bars per model (legend shows Spearman rho between num DEGs and mean DE-F1)
# for color, (model, scores) in zip(colors, model_scores.items()):
#     means = pd.Series(scores.mean(axis=0), index=perturbations)
#     # rho = means.corr(degs, method="spearman")
#     # fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
#     #                      name=f"{model} (ρ = {rho:.2f})", marker_color=color), row=1, col=1)

#     fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
#                          name=f"{model}", marker_color=color), row=1, col=1)


# # Bottom: perturbation magnitude (number of DEGs)
# fig.add_trace(go.Scatter(x=perturbations, y=degs, mode="lines+markers+text", text=degs, textposition="top center",
#                          line=dict(color="black"), name="# DEGs", showlegend=False), row=2, col=1)

# # fig.update_layout(barmode="group", title="DE-F1 Score mean (↑) by Model and Perturbation vs. Number of DEGs",
# #                   legend_title_text="Model (Spearman ρ with # DEGs)", font=dict(size=12))

# fig.update_layout(barmode="group", title="DE-F1 Score mean (↑) by Model and Perturbation vs. Number of DEGs",
#                   legend_title_text="Model", font=dict(size=12))

# # fig.update_layout(legend=dict(
# #     yanchor="top",
# #     y=0.99,
# #     xanchor="left",
# #     x=0.01
# # ))

# fig.update_layout(legend=dict(
#     orientation="h",
#     yanchor="bottom",
#     y=1.02,
#     xanchor="left",
#     x=0.01
# ))

# fig.update_yaxes(title_text="DE-F1 Score", row=1, col=1)
# fig.update_yaxes(title_text="# DEGs", range=[0, 50], row=2, col=1) # * degs.max() * 1.5
# fig.update_xaxes(title_text="Perturbation", row=2, col=1)

# # Tilt x-axis labels for better readability
# fig.update_xaxes(tickangle=25, row=2, col=1)

# # Save the figure as a PNG file
# fig.write_image("thesis_plots/results/seen/de_f1_score_by_model_and_perturbation.png", scale=3, width=1400, height=1000)

# fig.show()

### MSE

In [75]:
# RF
rf_mse_by_pert = []
for idx, row in df_rf_seen.iterrows():
    rf_mse_by_pert.append(ast.literal_eval(row["mse_test_loss_all"]))
rf_mse_by_pert = np.array(rf_mse_by_pert)

# Transformer
transformer_mse_by_pert = []
for idx, row in df_transformer_seen.iterrows():
    transformer_mse_by_pert.append(ast.literal_eval(row["mse_test_loss_all"]))
transformer_mse_by_pert = np.array(transformer_mse_by_pert)

# TF
tf_mse_by_pert = []
for idx, row in df_tf_seen.iterrows():
    tf_mse_by_pert.append(ast.literal_eval(row["mse_test_loss_all"]))
tf_mse_by_pert = np.array(tf_mse_by_pert)

# Mean Baseline
mean_baseline_mse_by_pert = []
for idx, row in df_mean_baseline_seen.iterrows():
    mean_baseline_mse_by_pert.append(ast.literal_eval(row["MSE_all"]))
mean_baseline_mse_by_pert = np.array(mean_baseline_mse_by_pert)

# Mean Shift Baseline
mean_shift_baseline_mse_by_pert = []
for idx, row in df_mean_shift_baseline_seen.iterrows():
    mean_shift_baseline_mse_by_pert.append(ast.literal_eval(row["MSE_all"]))
mean_shift_baseline_mse_by_pert = np.array(mean_shift_baseline_mse_by_pert)

In [76]:
df_bar_des_mse_dict = {"Model": ["Transformer", "Rectified Flow", "TransformerFlow", "Mean Baseline", "Mean Shift Baseline"]}

for i, pert in enumerate(perturbations):
    df_bar_des_mse_dict[pert] = [transformer_mse_by_pert[:, i].mean().item(), rf_mse_by_pert[:, i].mean().item(), tf_mse_by_pert[:, i].mean().item(), mean_baseline_mse_by_pert[:, i].mean().item(), mean_shift_baseline_mse_by_pert[:, i].mean().item()]

df_bar_des_mse = pd.DataFrame(df_bar_des_mse_dict)

df_bar_des_mse_melted = df_bar_des_mse.melt(id_vars="Model", var_name="Perturbation", value_name="MSE Loss")
fig = px.bar(df_bar_des_mse_melted, x="Perturbation", y="MSE Loss", color="Model", barmode="group", title="MSE Loss mean (↓) by Model and Perturbation")
# error bars
fig.update_traces(error_y=dict(type='data', array=[transformer_mse_by_pert[:, i].std() for i in range(len(perturbations))] + [rf_mse_by_pert[:, i].std() for i in range(len(perturbations))] + [tf_mse_by_pert[:, i].std() for i in range(len(perturbations))] + [mean_baseline_mse_by_pert[:, i].std() for i in range(len(perturbations))] + [mean_shift_baseline_mse_by_pert[:, i].std() for i in range(len(perturbations))], visible=True))
fig.show()

In [77]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

model_scores = {
    "Transformer": transformer_mse_by_pert,
    "Rectified Flow": rf_mse_by_pert,
    "TransformerFlow": tf_mse_by_pert,
    "Mean Baseline": mean_baseline_mse_by_pert,
    "Mean Shift Baseline": mean_shift_baseline_mse_by_pert,
}

colors = px.colors.qualitative.Plotly
# degs = pd.Series([num_degs[p] for p in perturbations], index=perturbations)
mse = pd.Series([np.round(mse_loss_by_pert[p], 4).item() for p in perturbations], index=perturbations)


fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.75, 0.25]) # 72 28

# Top: Energy bars per model
for color, (model, scores) in zip(colors, model_scores.items()):
    means = pd.Series(scores.mean(axis=0), index=perturbations)
    rho = means.corr(degs, method="spearman")
    # fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
    #                      name=f"{model} (ρ = {rho:.2f})", marker_color=color), row=1, col=1)
    print(f"{model}: Spearman ρ = {rho:.2f}")

    fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
                         name=f"{model}", marker_color=color), row=1, col=1)

    # Set y-axis to log scale for mse
    # fig.update_yaxes(type="log", row=1, col=1, tickvals=[0, 0.01, 0.1, 0.5], ticktext=['0', '0.01', '0.1', '0.5']) #tickvals=[0.01, 0.05, 0.1, 0.5, 1, 5], ticktext=['0.01', '0.05', '0.1', '0.5', '1', '5'])
             

# Bottom: perturbation magnitude 
fig.add_trace(go.Scatter(x=perturbations, y=mse, mode="lines+markers+text", text=mse, textposition="top center",
                         line=dict(color="black"), name="MSE", showlegend=False), row=2, col=1)

# fig.update_layout(barmode="group", title="DE-F1 Score mean (↑) by Model and Perturbation vs. Number of DEGs",
#                   legend_title_text="Model (Spearman ρ with # DEGs)", font=dict(size=12))

fig.update_layout(barmode="group", title="MSE (↓) by Model and Perturbation vs. MSE of Perturbation",
                  legend_title_text="Model", font=dict(size=25))

# fig.update_layout(legend=dict(
#     yanchor="top",
#     y=0.99,
#     xanchor="left",
#     x=0.01
# ))

fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.0,
    xanchor="left",
    x=0.01
))

fig.update_yaxes(title_text="MSE", row=1, col=1)
fig.update_yaxes(title_text="MSE", range=[0, 0.3], row=2, col=1) # * degs.max() * 1.5
fig.update_xaxes(title_text="Perturbation", row=2, col=1)

# Tilt x-axis labels for better readability
fig.update_xaxes(tickangle=25, row=2, col=1)

# Save the figure as a PNG file
fig.write_image("thesis_plots/results/seen/mse_by_model_and_perturbation.png", scale=5, width=1400, height=1000)

fig.show()

Transformer: Spearman ρ = 0.17
Rectified Flow: Spearman ρ = 0.02
TransformerFlow: Spearman ρ = -0.15
Mean Baseline: Spearman ρ = 0.05
Mean Shift Baseline: Spearman ρ = 0.05


## Mean prediction by model

In [ ]:
# df_baselines_seen = df_baselines[df_baselines["task"] == "SEEN"].reset_index(drop=True)
# df_mean_baseline_seen = df_baselines_seen[df_baselines_seen["baseline"] == "mean"].reset_index(drop=True)
# df_mean_shift_baseline_seen = df_baselines_seen[df_baselines_seen["baseline"] == "mean_shift"].reset_index(drop=True)

# df_baselines_unseen = df_baselines[df_baselines["task"] == "UNSEEN-PERT"].reset_index(drop=True)
# df_mean_baseline_unseen = df_baselines_unseen[df_baselines_unseen["baseline"] == "mean"].reset_index(drop=True)
# df_mean_shift_baseline_unseen = df_baselines_unseen[df_baselines_unseen["baseline"] == "mean_shift"].reset_index(drop=True)

In [ ]:
df_mean_baseline_seen.head()

,task,baseline,seed,de_test_score,de_precision_test_score,de_recall_test_score,energy_test_score,mse_test_loss,DE-F1_all,DE-Precision_all,DE-Recall_all,Energy_Loss_all,MSE_all
0,SEEN,mean,123,0.090914,0.048854,0.953367,3.779993,0.165294,"[0.22546972860125258, 0.1553398058252427, 0.06...","[0.1276595744680851, 0.08465608465608465, 0.03...","[0.9642857142857143, 0.9411764705882353, 0.928...","[3.8670806884765625, 3.722566604614258, 3.8482...","[0.1724759340286255, 0.16065968573093414, 0.17..."
2,SEEN,mean,456,0.094554,0.050751,0.932863,3.776072,0.164987,"[0.22510822510822512, 0.10178117048346057, 0.1...","[0.12807881773399016, 0.053763440860215055, 0....","[0.9285714285714286, 0.9523809523809523, 0.928...","[3.8562231063842773, 3.6941232681274414, 3.845...","[0.17177557945251465, 0.15789854526519775, 0.1..."
4,SEEN,mean,789,0.092451,0.049342,0.905074,3.782716,0.165635,"[0.2075055187637969, 0.09718670076726343, 0.09...","[0.11691542288557213, 0.051351351351351354, 0....","[0.9215686274509803, 0.9047619047619048, 0.909...","[3.8467798233032227, 3.7242918014526367, 3.853...","[0.17087426781654358, 0.16151945292949677, 0.1..."
6,SEEN,mean,101112,0.104726,0.055998,0.924127,3.786110,0.165924,"[0.18723404255319148, 0.13930348258706468, 0.1...","[0.10377358490566038, 0.07526881720430108, 0.0...","[0.9565217391304348, 0.9333333333333333, 0.896...","[3.8780927658081055, 3.7017745971679688, 3.878...","[0.1736830472946167, 0.15924407541751862, 0.17..."
8,SEEN,mean,131415,0.096255,0.051527,0.901289,3.784417,0.165801,"[0.20556745182012848, 0.09844559585492228, 0.1...","[0.11538461538461539, 0.05234159779614325, 0.0...","[0.9411764705882353, 0.8260869565217391, 0.888...","[3.8969926834106445, 3.7258338928222656, 3.854...","[0.1757672131061554, 0.16087505221366882, 0.17..."


### Save mean and std results as csv

In [ ]:
# Bar Chart models and baselines DE-F1
df_bar_des_baselines_dict = {"Model": ["Transformer", "Rectified Flow", "TransformerFlow", "Mean Baseline", "Mean Shift Baseline"]}

for metric in ["de_test_score", "de_precision_test_score", "de_recall_test_score", "energy_test_score", "mse_test_loss"]:
    df_bar_des_baselines_dict[metric] = [
        np.round(df_transformer_seen[metric].mean(), 4).item(),
        np.round(df_rf_seen[metric].mean(), 4).item(),
        np.round(df_tf_seen[metric].mean(), 4).item(),
        np.round(df_mean_baseline_seen[metric].mean(), 4).item(),
        np.round(df_mean_shift_baseline_seen[metric].mean(), 4).item()
    ]

df_bar_des_baselines = pd.DataFrame(df_bar_des_baselines_dict)
df_bar_des_baselines.to_csv("thesis_plots/results/seen/seen_results_mean.csv", index=False)

In [ ]:
# Standard deviation of DE-F1 Score by Model and Baseline
df_bar_des_baselines_std_dict = {"Model": ["Transformer", "Rectified Flow", "TransformerFlow", "Mean Baseline", "Mean Shift Baseline"]} 

for metric in ["de_test_score", "de_precision_test_score", "de_recall_test_score", "energy_test_score", "mse_test_loss"]:
    df_bar_des_baselines_std_dict[metric] = [
        np.round(df_transformer_seen[metric].std(), 4).item(),
        np.round(df_rf_seen[metric].std(), 4).item(),
        np.round(df_tf_seen[metric].std(), 4).item(),
        np.round(df_mean_baseline_seen[metric].std(), 4).item(),
        np.round(df_mean_shift_baseline_seen[metric].std(), 4).item()
    ]

df_bar_des_baselines_std = pd.DataFrame(df_bar_des_baselines_std_dict)
df_bar_des_baselines_std.to_csv("thesis_plots/results/seen/seen_results_std.csv", index=False)

In [ ]:
# Bar Chart models and baselines DE-F1
df_bar_des_baselines_unseen_dict = {"Model": ["Transformer", "Rectified Flow", "TransformerFlow", "Mean Baseline", "Mean Shift Baseline"]}

for metric in ["de_test_score", "de_precision_test_score", "de_recall_test_score", "energy_test_score", "mse_test_loss"]:
    df_bar_des_baselines_unseen_dict[metric] = [
        np.round(df_transformer_unseen[metric].mean(), 4).item(),
        np.round(df_rf_unseen[metric].mean(), 4).item(),
        np.round(df_tf_unseen[metric].mean(), 4).item(),
        np.round(df_mean_baseline_unseen[metric].mean(), 4).item(),
        np.round(df_mean_shift_baseline_unseen[metric].mean(), 4).item()
    ]

df_bar_des_baselines_unseen = pd.DataFrame(df_bar_des_baselines_unseen_dict)
df_bar_des_baselines_unseen.to_csv("thesis_plots/results/unseen/unseen_results_mean.csv", index=False)

In [ ]:
# Standard deviation of DE-F1 Score by Model and Baseline
df_bar_des_baselines_std_dict = {"Model": ["Transformer", "Rectified Flow", "TransformerFlow", "Mean Baseline", "Mean Shift Baseline"]}

for metric in ["de_test_score", "de_precision_test_score", "de_recall_test_score", "energy_test_score", "mse_test_loss"]:
    df_bar_des_baselines_std_dict[metric] = [
        np.round(df_transformer_unseen[metric].std(), 4).item(),
        np.round(df_rf_unseen[metric].std(), 4).item(),
        np.round(df_tf_unseen[metric].std(), 4).item(),
        np.round(df_mean_baseline_unseen[metric].std(), 4).item(),
        np.round(df_mean_shift_baseline_unseen[metric].std(), 4).item()
    ]

df_bar_des_baselines_std_unseen = pd.DataFrame(df_bar_des_baselines_std_dict)
df_bar_des_baselines_std_unseen.to_csv("thesis_plots/results/unseen/unseen_results_std.csv", index=False)

In [ ]:
df_bar_des_baselines

,Model,de_test_score,de_precision_test_score,de_recall_test_score,energy_test_score,mse_test_loss
0,Transformer,0.6097,0.8013,0.5161,0.0386,0.3436
1,Rectified Flow,0.5091,0.6930,0.4440,0.0543,0.3294
2,TransformerFlow,0.4634,0.6218,0.4061,0.0541,0.3314
3,Mean Baseline,0.0945,0.0507,0.9129,3.7832,0.1656
4,Mean Shift Baseline,0.1226,0.0666,0.9611,0.0306,0.3356


In [ ]:
df_bar_des_baselines.iloc[:, :2]

,Model,de_test_score
0,Transformer,0.6097
1,Rectified Flow,0.5091
2,TransformerFlow,0.4634
3,Mean Baseline,0.0945
4,Mean Shift Baseline,0.1226


In [ ]:
# Bar Plot for DE-F1 Score with Baselines
metric_map = {
    "de_test_score": "DE-F1 Score",
    "de_precision_test_score": "DE-Precision Score",
    "de_recall_test_score": "DE-Recall Score",
    "energy_test_score": "Energy Score",
    "mse_test_loss": "MSE Loss"
}
for metric in ["de_test_score", "de_precision_test_score", "de_recall_test_score", "energy_test_score", "mse_test_loss"]:
    log_y = False
    if metric == "energy_test_score" or metric == "mse_test_loss":
        direction = "↓"
    else:
        direction = "↑"
    if metric != "energy_test_score":
        fig = px.bar(df_bar_des_baselines, x="Model", y=metric, color="Model", log_y=log_y, title=f"{metric_map[metric]} mean ({direction}) by Model and Baseline")
        fig.write_image(f"thesis_plots/results/seen/{metric}.png", scale=2)
        fig.show()
    else:
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots

        categories = df_bar_des_baselines["Model"]
        values = df_bar_des_baselines[metric]
        color_sequence = px.colors.qualitative.Plotly
        # 2. Create stacked subplots (2 rows, 1 column)
        # row_heights=[top_half, bottom_half] allocates space for the break
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.15, row_heights=[0.4, 0.6])

        # 3. Add the data to BOTH subplots
        # fig.add_trace(go.Bar(x=categories, y=values, marker_color='#AB63FA', legendgroup="Model", showlegend=False), row=1, col=1)
        # fig.add_trace(go.Bar(x=categories, y=values, marker_color=['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A'], legendgroup="Model", showlegend=False), row=2, col=1)

        model_colors = {m: color_sequence[i % len(color_sequence)] for i, m in enumerate(categories)}

        for model in categories:
            val = df_bar_des_baselines.loc[df_bar_des_baselines["Model"] == model, metric].values[0]
            fig.add_trace(
                go.Bar(x=[model], y=[val], name=model, marker_color=model_colors[model],
                    legendgroup=model, showlegend=False),
                row=1, col=1,
            )
            fig.add_trace(
                go.Bar(x=[model], y=[val], name=model, marker_color=model_colors[model],
                    legendgroup=model, showlegend=True),
                row=2, col=1,
            )

        # 4. Set the different Y-axis ranges (this creates the "cut-off" gap)
        # fig.update_yaxes(range=[3, 5], row=1, col=1) # Top plot: zoom in on the outlier peak
        # fig.update_yaxes(range=[0, 0.1], row=2, col=1)   # Bottom plot: zoom in on normal values
        fig.update_yaxes(range=[3, 5], nticks=3, showgrid=True, zeroline=True, row=1, col=1)
        fig.update_yaxes(range=[0, 0.1], nticks=3, showgrid=True, zeroline=True, row=2, col=1)
        fig.update_xaxes(showticklabels=False, row=1, col=1)

        # 5. Academic styling adjustments
        # direciton = "↓" if metric in ["energy_test_score", "mse_test_loss"] else "↑"
        # fig.update_layout(
        #     template='plotly',
        #     title_text=f"{metric_map[metric]} mean ({direction}) by Model and Baseline",
        #     # height=500,
        #     # width=600
        # )
        # # Add Legend
        # fig.update_layout(showlegend=True, )#legend=dict(title="Model", orientation="h", yanchor="top", y=0, xanchor="right", x=0))

        # Clean up axes so it looks like one cohesive chart
        # fig.update_xaxes(showticklabels=False, row=1, col=1) # Hide X-axis on the top plot

        # Optional: Add text on top of the broken gap to explicitly point out the break
        # fig.add_annotation(dict(x=0.5, y=0.82, xref="paper", yref="paper", text="// Axis Break //", showarrow=False, font=dict(size=12, color="gray")))

        fig.update_layout(
            title_text=f"{metric_map[metric]} mean ({direction}) by Model and Baseline",
            # height=500,
            # width=600,
            barmode="group",
            legend=dict(title="Model"),
        )

        fig.update_layout(
            legend=dict(title="Model", tracegroupgap=0)
        )

        fig.add_annotation(
            x=0.02, y=0.59, xref="paper", yref="paper",
            text="// Axis Break //", showarrow=False, font=dict(size=12, color="gray"),
        )

        fig.write_image(f"thesis_plots/results/seen/{metric}.png", scale=2)
        fig.show()

        # fig.show()

    # if metric == "energy_test_score":
    #     fig.update_yaxes(type="log", range=[-2.0 ,1.0])
    
# df_bar_des_baselines_melted = df_bar_des_baselines.iloc[:, :].melt(id_vars="Model", var_name="Metric", value_name="Score")
# fig = px.bar(df_bar_des_baselines_melted, x="Metric", y="Score", color="Model", barmode="group", title="Score mean (↑ for DE-F1, DE-Precision, DE-Recall; ↓ for Energy, MSE) by Model and Baseline")
# fig.show()



# Unseen Perturbations

In [78]:
# df_unseen_reflow = df_unseen_reflow[cols_of_interest + val_metric_cols + test_metric_cols + all_values_cols]
# df_unseen_reflow.head()

In [79]:
# df_unseen = df_unseen_reflow.where(df_unseen_reflow["reflow"] == 0).dropna()
# df_unseen.head()

21 per perturbation because ran seed 123 twice

In [80]:
# df_rf_unseen = df_unseen[df_unseen["model_family"] == "RectifiedFlow"][1:]
# df_tf_unseen = df_unseen[df_unseen["model_family"] == "TransformerFlow"][1:]
# df_transformer_unseen = df_unseen[df_unseen["model_family"] == "Transformer"][1:]

In [81]:
# df_rf_unseen

In [82]:
np.round(df_transformer_unseen.describe(), 4)

,seed,de_val_score,de_precision_val_score,de_recall_val_score,energy_val_score,mse_val_loss,de_test_score,de_precision_test_score,de_recall_test_score,energy_test_score,mse_test_loss
count,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000
mean,292074.0000,0.5692,0.7913,0.4546,0.0291,0.3434,0.4618,0.7003,0.3533,0.0801,0.3475
std,188456.3373,0.0296,0.0347,0.0327,0.0015,0.0043,0.0331,0.0275,0.0383,0.0096,0.0044
min,123.0000,0.5020,0.7070,0.3936,0.0263,0.3321,0.3674,0.6502,0.2533,0.0630,0.3398
25%,154142.2500,0.5470,0.7785,0.4342,0.0280,0.3424,0.4431,0.6845,0.3305,0.0743,0.3440
50%,298081.5000,0.5718,0.7878,0.4547,0.0290,0.3444,0.4613,0.6960,0.3523,0.0782,0.3467
75%,442020.7500,0.5917,0.8168,0.4809,0.0302,0.3455,0.4855,0.7116,0.3795,0.0845,0.3513
max,585960.0000,0.6158,0.8631,0.5343,0.0321,0.3508,0.5147,0.7720,0.4310,0.0996,0.3557


In [83]:
np.round(df_rf_unseen.describe(), 4)

,seed,de_val_score,de_precision_val_score,de_recall_val_score,energy_val_score,mse_val_loss,de_test_score,de_precision_test_score,de_recall_test_score,energy_test_score,mse_test_loss
count,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000
mean,292074.0000,0.4550,0.7041,0.3650,0.0423,0.3341,0.4348,0.5970,0.3611,0.0719,0.3389
std,188456.3373,0.0354,0.1007,0.0332,0.0059,0.0033,0.0276,0.0400,0.0324,0.0057,0.0027
min,123.0000,0.3733,0.4942,0.2995,0.0365,0.3265,0.3698,0.5202,0.3133,0.0643,0.3347
25%,154142.2500,0.4426,0.6450,0.3376,0.0380,0.3323,0.4187,0.5715,0.3319,0.0672,0.3374
50%,298081.5000,0.4579,0.7161,0.3695,0.0404,0.3344,0.4327,0.5907,0.3601,0.0705,0.3386
75%,442020.7500,0.4854,0.7840,0.3887,0.0440,0.3361,0.4503,0.6219,0.3769,0.0761,0.3408
max,585960.0000,0.5089,0.8516,0.4143,0.0571,0.3413,0.4895,0.6727,0.4228,0.0852,0.3433


In [84]:
np.round(df_tf_unseen.describe(), 4)

,seed,de_val_score,de_precision_val_score,de_recall_val_score,energy_val_score,mse_val_loss,de_test_score,de_precision_test_score,de_recall_test_score,energy_test_score,mse_test_loss
count,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000
mean,292074.0000,0.4297,0.5806,0.3737,0.0420,0.3327,0.4569,0.4337,0.4979,0.1080,0.3373
std,188456.3373,0.0312,0.0633,0.0353,0.0026,0.0029,0.0289,0.0304,0.0383,0.0113,0.0050
min,123.0000,0.3727,0.4042,0.3225,0.0387,0.3270,0.4021,0.3775,0.3888,0.0941,0.3267
25%,154142.2500,0.4028,0.5513,0.3530,0.0399,0.3312,0.4412,0.4174,0.4852,0.0987,0.3347
50%,298081.5000,0.4364,0.5856,0.3727,0.0421,0.3325,0.4620,0.4342,0.5049,0.1057,0.3373
75%,442020.7500,0.4539,0.6233,0.4052,0.0434,0.3341,0.4788,0.4529,0.5199,0.1142,0.3407
max,585960.0000,0.4822,0.6772,0.4358,0.0481,0.3381,0.5013,0.4941,0.5537,0.1387,0.3467


In [85]:
# Unseen Perturbations


## Analysis by perturbation

In [86]:
unseen_perturbations = ["RMC-6236", "4EGI-1", "AT7519"]

In [87]:
df_transformer_unseen.head()

,Name,seed,model_family,task,de_val_score,de_precision_val_score,de_recall_val_score,energy_val_score,mse_val_loss,de_test_score,de_precision_test_score,de_recall_test_score,energy_test_score,mse_test_loss,de_test_score_all,de_precition_test_score_all,de_recall_test_score_all,energy_test_score_all,mse_test_loss_all
20,TRAINUNSEEN-PERT-Transformer-seed585960-serene...,585960,Transformer,UNSEEN-PERT,0.584506,0.823099,0.458796,0.028257,0.343278,0.367402,0.689236,0.253287,0.098706,0.346302,"[0.3404255319148936,0.2617801047120419,0.5]","[0.7804878048780488,0.4807692307692308,0.80645...","[0.21768707482993196,0.17985611510791366,0.362...","[0.13873672485351562,0.1182270050048828,0.0391...","[0.34886905550956726,0.34129300713539124,0.348..."
21,TRAINUNSEEN-PERT-Transformer-seed555657-expert...,555657,Transformer,UNSEEN-PERT,0.601103,0.823334,0.482758,0.027709,0.345055,0.458506,0.700678,0.343686,0.078416,0.351269,"[0.5161290322580645,0.39,0.4693877551020408]","[0.9032258064516128,0.5416666666666666,0.65714...","[0.36129032258064514,0.3046875,0.3650793650793...","[0.07869148254394531,0.1232767105102539,0.0332...","[0.34803423285484314,0.34938114881515503,0.356..."
22,TRAINUNSEEN-PERT-Transformer-seed525354-glad-g...,525354,Transformer,UNSEEN-PERT,0.547396,0.819225,0.416030,0.027854,0.344318,0.463727,0.677540,0.353299,0.066173,0.344138,"[0.6083333333333334,0.2458100558659218,0.53703...","[0.8488372093023255,0.4,0.7837837837837838]","[0.474025974025974,0.1774193548387097,0.408450...","[0.05791187286376953,0.1086273193359375,0.0319...","[0.34559789299964905,0.3451177775859833,0.3416..."
23,TRAINUNSEEN-PERT-Transformer-seed495051-driven...,495051,Transformer,UNSEEN-PERT,0.577549,0.800243,0.453758,0.029262,0.341074,0.470846,0.687879,0.366643,0.075278,0.339835,"[0.4615384615384616,0.32978723404255317,0.6212...","[0.8181818181818182,0.5,0.7454545454545455]","[0.32142857142857145,0.246031746031746,0.53246...","[0.07132816314697266,0.12020206451416016,0.034...","[0.34182482957839966,0.3374529778957367,0.3402..."
24,TRAINUNSEEN-PERT-Transformer-seed464748-distin...,464748,Transformer,UNSEEN-PERT,0.611388,0.863135,0.484226,0.028046,0.346137,0.436304,0.650189,0.340814,0.073429,0.343543,"[0.5084745762711864,0.25806451612903225,0.5423...","[0.8823529411764706,0.4528301886792453,0.61538...","[0.35714285714285715,0.1804511278195489,0.4848...","[0.07606315612792969,0.11044883728027344,0.033...","[0.3436427116394043,0.3413812220096588,0.34560..."


### DE-F1

In [88]:
# RF
rf_de_by_pert_unseen = []
for idx, row in df_rf_unseen.iterrows():
    rf_de_by_pert_unseen.append(ast.literal_eval(row["de_test_score_all"]))
rf_de_by_pert_unseen = np.array(rf_de_by_pert_unseen)

# Transformer
transformer_de_by_pert_unseen = []
for idx, row in df_transformer_unseen.iterrows():
    transformer_de_by_pert_unseen.append(ast.literal_eval(row["de_test_score_all"]))
transformer_de_by_pert_unseen = np.array(transformer_de_by_pert_unseen)

# TF
tf_de_by_pert_unseen = []
for idx, row in df_tf_unseen.iterrows():
    tf_de_by_pert_unseen.append(ast.literal_eval(row["de_test_score_all"]))
tf_de_by_pert_unseen = np.array(tf_de_by_pert_unseen)

# Mean Baseline
mean_baseline_de_by_pert_unseen = []
for idx, row in df_mean_baseline_unseen.iterrows():
    mean_baseline_de_by_pert_unseen.append(ast.literal_eval(row["DE-F1_all"]))
mean_baseline_de_by_pert_unseen = np.array(mean_baseline_de_by_pert_unseen)


# Mean Shift Baseline
mean_shift_baseline_de_by_pert_unseen = []
for idx, row in df_mean_shift_baseline_unseen.iterrows():
    mean_shift_baseline_de_by_pert_unseen.append(ast.literal_eval(row["DE-F1_all"]))
mean_shift_baseline_de_by_pert_unseen = np.array(mean_shift_baseline_de_by_pert_unseen)


print(rf_de_by_pert_unseen.mean(axis=0))
print(transformer_de_by_pert_unseen.mean(axis=0))
print(tf_de_by_pert_unseen.mean(axis=0))

[0.52314085 0.29998389 0.48121321]
[0.51341948 0.3159361  0.55614984]
[0.55700817 0.37579612 0.43784159]


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

model_scores = {
    "Transformer": transformer_de_by_pert_unseen,
    "Rectified Flow": rf_de_by_pert_unseen,
    "TransformerFlow": tf_de_by_pert_unseen,
    "Mean Baseline": mean_baseline_de_by_pert_unseen,
    "Mean Shift Baseline": mean_shift_baseline_de_by_pert_unseen,
}
colors = px.colors.qualitative.Plotly
degs = pd.Series([num_degs_unseen[p] for p in unseen_perturbations], index=unseen_perturbations)
edistance = pd.Series([np.round(energy_loss_by_pert_unseen[p], 4).item() for p in unseen_perturbations], index=unseen_perturbations)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.75, 0.25], specs=[[{}], [{"secondary_y": True}]]) # 72 28

# Top: DE-F1 bars per model (legend shows Spearman rho between num DEGs and mean DE-F1)
for color, (model, scores) in zip(colors, model_scores.items()):
    means = pd.Series(scores.mean(axis=0), index=unseen_perturbations)
    # rho = means.corr(degs, method="spearman")
    # fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
    #                      name=f"{model} (ρ = {rho:.2f})", marker_color=color), row=1, col=1)


    fig.add_trace(go.Bar(x=unseen_perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
                         name=f"{model}", marker_color=color), row=1, col=1)

# Bottom: perturbation magnitude (number of DEGs)
fig.add_trace(go.Scatter(x=unseen_perturbations, y=degs, mode="lines+markers+text", text=degs, textposition="bottom left",
                         line=dict(color="red"), name="# DEGs", showlegend=False), row=2, col=1, secondary_y=True)
fig.add_trace(go.Scatter(x=unseen_perturbations, y=edistance, mode="lines+markers+text", text=edistance, textposition="top center",
                         line=dict(color="black"), name="E-distance", showlegend=False), row=2, col=1, secondary_y=False)

# fig.update_layout(barmode="group", title="DE-F1 Score mean (↑) by Model and Perturbation vs. Number of DEGs",
#                   legend_title_text="Model (Spearman ρ with # DEGs)", font=dict(size=12))

# fig.update_layout(barmode="group", title="DE-F1 Score mean (↑) by Model and Unseen Perturbation vs. Number of DEGs",
#                   legend_title_text="Model", font=dict(size=25))
fig.update_layout(barmode="group", title="DE-F1 Score mean (↑) by Model and Unseen Pert. vs. E-distance of Pert.",
                  legend_title_text="Model", font=dict(size=25))

# fig.update_layout(legend=dict(
#     yanchor="top",
#     y=0.99,
#     xanchor="left",
#     x=0.01
# ))

fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.0,
    xanchor="left",
    x=0.01,
    itemsizing="constant",
))

fig.update_yaxes(title_text="DE-F1 Score", row=1, col=1)
fig.update_yaxes(title_text="# DEGs", range=[-10, 50], row=2, col=1, secondary_y=True, title_font_color="red") # * degs.max() * 1.5
fig.update_yaxes(title_text="E-distance", range=[-0.2, 1], row=2, col=1, secondary_y=False) # * degs.max() * 1.5
fig.update_xaxes(title_text="Perturbation", row=2, col=1)

# Tilt x-axis labels for better readability
fig.update_xaxes(tickangle=25, row=2, col=1)

fig.update_yaxes(tickvals=[0, 10, 20, 30, 40, 50], row=2, col=1, secondary_y=True, tickfont_color="red")
fig.update_yaxes(tickvals=[0, 0.2, 0.4, 0.6, 0.8, 1], row=2, col=1, secondary_y=False)

fig.show()

fig.update_layout(
    font=dict(
        size=25
    )
)
# Save the figure as a PNG file
fig.write_image("thesis_plots/results/unseen/de_f1_score_by_model_and_perturbation_unseen.png", scale=5, width=1400, height=1000)


### DE-F1 Precision

In [42]:
# RF
rf_de_precision_by_pert_unseen = []
for idx, row in df_rf_unseen.iterrows():
    rf_de_precision_by_pert_unseen.append(ast.literal_eval(row["de_precition_test_score_all"]))
rf_de_precision_by_pert_unseen = np.array(rf_de_precision_by_pert_unseen)

# Transformer
transformer_de_precision_by_pert_unseen = []
for idx, row in df_transformer_unseen.iterrows():
    transformer_de_precision_by_pert_unseen.append(ast.literal_eval(row["de_precition_test_score_all"]))
transformer_de_precision_by_pert_unseen = np.array(transformer_de_precision_by_pert_unseen)

# TF
tf_de_precision_by_pert_unseen = []
for idx, row in df_tf_unseen.iterrows():
    tf_de_precision_by_pert_unseen.append(ast.literal_eval(row["de_precition_test_score_all"]))
tf_de_precision_by_pert_unseen = np.array(tf_de_precision_by_pert_unseen)

# Mean Baseline
mean_baseline_de_precision_by_pert_unseen = []
for idx, row in df_mean_baseline_unseen.iterrows():
    mean_baseline_de_precision_by_pert_unseen.append(ast.literal_eval(row["DE-Precision_all"]))
mean_baseline_de_precision_by_pert_unseen = np.array(mean_baseline_de_precision_by_pert_unseen)

# Mean Shift Baseline
mean_shift_baseline_de_precision_by_pert_unseen = []
for idx, row in df_mean_shift_baseline_unseen.iterrows():
    mean_shift_baseline_de_precision_by_pert_unseen.append(ast.literal_eval(row["DE-Precision_all"]))
mean_shift_baseline_de_precision_by_pert_unseen = np.array(mean_shift_baseline_de_precision_by_pert_unseen)

In [43]:
# df_bar_des_precision_dict_unseen = {"Model": ["Transformer", "Rectified Flow", "TransformerFlow", "Mean Baseline", "Mean Shift Baseline"]}

# for i, pert in enumerate(unseen_perturbations):
#     df_bar_des_precision_dict_unseen[pert] = [transformer_de_precision_by_pert_unseen[:, i].mean().item(), rf_de_precision_by_pert_unseen[:, i].mean().item(), tf_de_precision_by_pert_unseen[:, i].mean().item(), mean_baseline_de_precision_by_pert_unseen[:, i].mean().item(), mean_shift_baseline_de_precision_by_pert_unseen[:, i].mean().item()]

# df_bar_des_precision_unseen = pd.DataFrame(df_bar_des_precision_dict_unseen)

# df_bar_des_precision_unseen_melted = df_bar_des_precision_unseen.melt(id_vars="Model", var_name="Perturbation", value_name="DE-Precision Score")
# fig = px.bar(df_bar_des_precision_unseen_melted, x="Perturbation", y="DE-Precision Score", color="Model", barmode="group", title="DE-Precision Score mean (↑) by Model and Perturbation")
# # error bars
# fig.update_traces(error_y=dict(type='data', array=[transformer_de_precision_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [rf_de_precision_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [tf_de_precision_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [mean_baseline_de_precision_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [mean_shift_baseline_de_precision_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))], visible=True))
# fig.show()


In [44]:

from plotly.subplots import make_subplots
import plotly.graph_objects as go

model_scores = {
    "Transformer": transformer_de_precision_by_pert_unseen,
    "Rectified Flow": rf_de_precision_by_pert_unseen,
    "TransformerFlow": tf_de_precision_by_pert_unseen,
    "Mean Baseline": mean_baseline_de_precision_by_pert_unseen,
    "Mean Shift Baseline": mean_shift_baseline_de_precision_by_pert_unseen,
}
colors = px.colors.qualitative.Plotly
degs = pd.Series([num_degs_unseen[p] for p in unseen_perturbations], index=unseen_perturbations)

fig = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[1.0]) # 72 28

# Top: DE-F1 bars per model (legend shows Spearman rho between num DEGs and mean DE-F1)
for color, (model, scores) in zip(colors, model_scores.items()):
    means = pd.Series(scores.mean(axis=0), index=unseen_perturbations)
    # rho = means.corr(degs, method="spearman")
    # fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
    #                      name=f"{model} (ρ = {rho:.2f})", marker_color=color), row=1, col=1)

    fig.add_trace(go.Bar(x=unseen_perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
                         name=f"{model}", marker_color=color), row=1, col=1)

# Bottom: perturbation magnitude (number of DEGs)
# fig.add_trace(go.Scatter(x=unseen_perturbations, y=degs, mode="lines+markers+text", text=degs, textposition="top center",
#                          line=dict(color="black"), name="# DEGs", showlegend=False), row=2, col=1)

# fig.update_layout(barmode="group", title="DE-F1 Score mean (↑) by Model and Perturbation vs. Number of DEGs",
#                   legend_title_text="Model (Spearman ρ with # DEGs)", font=dict(size=12))

fig.update_layout(barmode="group", title="DE Precision Score mean (↑) by Model and Unseen Perturbation",
                  legend_title_text="Model", font=dict(size=25))

# fig.update_layout(legend=dict(
#     yanchor="top",
#     y=0.99,
#     xanchor="left",
#     x=0.01
# ))

fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.0,
    xanchor="left",
    x=0.01,
    itemsizing="constant",
))

fig.update_yaxes(title_text="DE-Precision Score", row=1, col=1)
# fig.update_yaxes(title_text="# DEGs", range=[0, 30], row=2, col=1) # * degs.max() * 1.5
fig.update_xaxes(title_text="Perturbation", row=1, col=1)

# Tilt x-axis labels for better readability
fig.update_xaxes(tickangle=25, row=2, col=1)

fig.show()

fig.update_layout(
    font=dict(
        size=25
    )
)
# Save the figure as a PNG file
fig.write_image("thesis_plots/results/unseen/de_precision_by_model_and_perturbation_unseen.png", scale=5, width=1400, height=1000)


### DE-F1 Recall

In [45]:
# RF
rf_de_recall_by_pert_unseen = []
for idx, row in df_rf_unseen.iterrows():
    rf_de_recall_by_pert_unseen.append(ast.literal_eval(row["de_recall_test_score_all"]))
rf_de_recall_by_pert_unseen = np.array(rf_de_recall_by_pert_unseen)

# Transformer
transformer_de_recall_by_pert_unseen = []
for idx, row in df_transformer_unseen.iterrows():
    transformer_de_recall_by_pert_unseen.append(ast.literal_eval(row["de_recall_test_score_all"]))
transformer_de_recall_by_pert_unseen = np.array(transformer_de_recall_by_pert_unseen)

# TF
tf_de_recall_by_pert_unseen = []
for idx, row in df_tf_unseen.iterrows():
    tf_de_recall_by_pert_unseen.append(ast.literal_eval(row["de_recall_test_score_all"]))
tf_de_recall_by_pert_unseen = np.array(tf_de_recall_by_pert_unseen)

# Mean Baseline
mean_baseline_de_recall_by_pert_unseen = []
for idx, row in df_mean_baseline_unseen.iterrows():
    mean_baseline_de_recall_by_pert_unseen.append(ast.literal_eval(row["DE-Recall_all"]))
mean_baseline_de_recall_by_pert_unseen = np.array(mean_baseline_de_recall_by_pert_unseen)

# Mean Shift Baseline
mean_shift_baseline_de_recall_by_pert_unseen = []
for idx, row in df_mean_shift_baseline_unseen.iterrows():
    mean_shift_baseline_de_recall_by_pert_unseen.append(ast.literal_eval(row["DE-Recall_all"]))
mean_shift_baseline_de_recall_by_pert_unseen = np.array(mean_shift_baseline_de_recall_by_pert_unseen)

In [46]:
# df_bar_des_recall_dict_unseen = {"Model": ["Transformer", "Rectified Flow", "TransformerFlow", "Mean Baseline", "Mean Shift Baseline"]}

# for i, pert in enumerate(unseen_perturbations):
#     df_bar_des_recall_dict_unseen[pert] = [transformer_de_recall_by_pert_unseen[:, i].mean().item(), rf_de_recall_by_pert_unseen[:, i].mean().item(), tf_de_recall_by_pert_unseen[:, i].mean().item(), mean_baseline_de_recall_by_pert_unseen[:, i].mean().item(), mean_shift_baseline_de_recall_by_pert_unseen[:, i].mean().item()]

# df_bar_des_recall_unseen = pd.DataFrame(df_bar_des_recall_dict_unseen)

# df_bar_des_recall_unseen_melted = df_bar_des_recall_unseen.melt(id_vars="Model", var_name="Perturbation", value_name="DE-Recall Score")
# fig = px.bar(df_bar_des_recall_unseen_melted, x="Perturbation", y="DE-Recall Score", color="Model", barmode="group", title="DE-Recall Score mean (↑) by Model and Perturbation")
# # error bars
# fig.update_traces(error_y=dict(type='data', array=[transformer_de_recall_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [rf_de_recall_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [tf_de_recall_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [mean_baseline_de_recall_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [mean_shift_baseline_de_recall_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))], visible=True))
# fig.show()

In [47]:

from plotly.subplots import make_subplots
import plotly.graph_objects as go

model_scores = {
    "Transformer": transformer_de_recall_by_pert_unseen,
    "Rectified Flow": rf_de_recall_by_pert_unseen,
    "TransformerFlow": tf_de_recall_by_pert_unseen,
    "Mean Baseline": mean_baseline_de_recall_by_pert_unseen,
    "Mean Shift Baseline": mean_shift_baseline_de_recall_by_pert_unseen,
}
colors = px.colors.qualitative.Plotly
degs = pd.Series([num_degs_unseen[p] for p in unseen_perturbations], index=unseen_perturbations)

fig = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[1.0]) # 72 28

# Top: DE-F1 bars per model (legend shows Spearman rho between num DEGs and mean DE-F1)
for color, (model, scores) in zip(colors, model_scores.items()):
    means = pd.Series(scores.mean(axis=0), index=unseen_perturbations)
    # rho = means.corr(degs, method="spearman")
    # fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
    #                      name=f"{model} (ρ = {rho:.2f})", marker_color=color), row=1, col=1)

    fig.add_trace(go.Bar(x=unseen_perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
                         name=f"{model}", marker_color=color), row=1, col=1)

# Bottom: perturbation magnitude (number of DEGs)
# fig.add_trace(go.Scatter(x=unseen_perturbations, y=degs, mode="lines+markers+text", text=degs, textposition="top center",
#                          line=dict(color="black"), name="# DEGs", showlegend=False), row=2, col=1)

# fig.update_layout(barmode="group", title="DE-F1 Score mean (↑) by Model and Perturbation vs. Number of DEGs",
#                   legend_title_text="Model (Spearman ρ with # DEGs)", font=dict(size=12))

fig.update_layout(barmode="group", title="DE Recall Score mean (↑) by Model and Unseen Perturbation",
                  legend_title_text="Model", font=dict(size=25))

# fig.update_layout(legend=dict(
#     yanchor="top",
#     y=0.99,
#     xanchor="left",
#     x=0.01
# ))

fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.0,
    xanchor="left",
    x=0.01,
    itemsizing="constant",
))

fig.update_yaxes(title_text="DE-Recall Score", row=1, col=1)
# fig.update_yaxes(title_text="# DEGs", range=[0, 30], row=2, col=1) # * degs.max() * 1.5
fig.update_xaxes(title_text="Perturbation", row=1, col=1)

# Tilt x-axis labels for better readability
fig.update_xaxes(tickangle=25, row=2, col=1)

fig.show()

fig.update_layout(
    font=dict(
        size=25
    )
)
# Save the figure as a PNG file
fig.write_image("thesis_plots/results/unseen/de_recall_by_model_and_perturbation_unseen.png", scale=5, width=1400, height=1000)


### Energy Score

In [48]:
def clean_edistance(string):
    if pd.isna(string):
        return string
    # 1. Strip away the "np.float64(...)" wrappers
    cleaned = re.sub(r"np\.float64\((.*?)\)", r"\1", str(string))
    # 2. Safely parse the remaining standard Python list string
    return ast.literal_eval(cleaned)

In [49]:
df_mean_baseline_unseen["Energy_Loss_all"] = df_mean_baseline_unseen["Energy_Loss_all"].apply(clean_edistance)
df_mean_shift_baseline_unseen["Energy_Loss_all"] = df_mean_shift_baseline_unseen["Energy_Loss_all"].apply(clean_edistance)

/tmp/ipykernel_3728144/393340939.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_mean_baseline_unseen["Energy_Loss_all"] = df_mean_baseline_unseen["Energy_Loss_all"].apply(clean_edistance)
/tmp/ipykernel_3728144/393340939.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_mean_shift_baseline_unseen["Energy_Loss_all"] = df_mean_shift_baseline_unseen["Energy_Loss_all"].apply(clean_edistance)


In [50]:
# RF 
rf_energy_by_pert_unseen = []
for idx, row in df_rf_unseen.iterrows():
    rf_energy_by_pert_unseen.append(ast.literal_eval(row["energy_test_score_all"]))
rf_energy_by_pert_unseen = np.array(rf_energy_by_pert_unseen)

# Transformer
transformer_energy_by_pert_unseen = []
for idx, row in df_transformer_unseen.iterrows():
    transformer_energy_by_pert_unseen.append(ast.literal_eval(row["energy_test_score_all"]))
transformer_energy_by_pert_unseen = np.array(transformer_energy_by_pert_unseen)

# TF
tf_energy_by_pert_unseen = []
for idx, row in df_tf_unseen.iterrows():
    tf_energy_by_pert_unseen.append(ast.literal_eval(row["energy_test_score_all"]))
tf_energy_by_pert_unseen = np.array(tf_energy_by_pert_unseen)

# Mean Baseline
mean_baseline_energy_by_pert_unseen = []
for idx, row in df_mean_baseline_unseen.iterrows():
    mean_baseline_energy_by_pert_unseen.append(row["Energy_Loss_all"])
mean_baseline_energy_by_pert_unseen = np.array(mean_baseline_energy_by_pert_unseen)

# Mean Shift Baseline
mean_shift_baseline_energy_by_pert_unseen = []
for idx, row in df_mean_shift_baseline_unseen.iterrows():
    mean_shift_baseline_energy_by_pert_unseen.append(row["Energy_Loss_all"])
mean_shift_baseline_energy_by_pert_unseen = np.array(mean_shift_baseline_energy_by_pert_unseen)

In [51]:
# df_bar_des_energy_dict_unseen = {"Model": ["Transformer", "Rectified Flow", "TransformerFlow", "Mean Baseline", "Mean Shift Baseline"]}

# for i, pert in enumerate(unseen_perturbations):
#     df_bar_des_energy_dict_unseen[pert] = [transformer_energy_by_pert_unseen[:, i].mean().item(), rf_energy_by_pert_unseen[:, i].mean().item(), tf_energy_by_pert_unseen[:, i].mean().item(), mean_baseline_energy_by_pert_unseen[:, i].mean().item(), mean_shift_baseline_energy_by_pert_unseen[:, i].mean().item()]

# df_bar_des_energy_unseen = pd.DataFrame(df_bar_des_energy_dict_unseen)

# df_bar_des_energy_unseen_melted = df_bar_des_energy_unseen.melt(id_vars="Model", var_name="Perturbation", value_name="Energy Score")
# fig = px.bar(df_bar_des_energy_unseen_melted, x="Perturbation", y="Energy Score", color="Model", barmode="group", log_y=True, title="Energy Score mean (↓) by Model and Perturbation")
# # error bars
# fig.update_traces(error_y=dict(type='data', array=[transformer_energy_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [rf_energy_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [tf_energy_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [mean_baseline_energy_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [mean_shift_baseline_energy_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))], visible=True))
# fig.show()

In [52]:

from plotly.subplots import make_subplots
import plotly.graph_objects as go

model_scores = {
    "Transformer": transformer_energy_by_pert_unseen,
    "Rectified Flow": rf_energy_by_pert_unseen,
    "TransformerFlow": tf_energy_by_pert_unseen,
    "Mean Baseline": mean_baseline_energy_by_pert_unseen,
    "Mean Shift Baseline": mean_shift_baseline_energy_by_pert_unseen,
}
colors = px.colors.qualitative.Plotly
edistance = pd.Series([energy_loss_by_pert_unseen[p] for p in unseen_perturbations], index=unseen_perturbations)


fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.75, 0.25]) # 72 28

# Top: DE-F1 bars per model (legend shows Spearman rho between num DEGs and mean DE-F1)
for color, (model, scores) in zip(colors, model_scores.items()):
    means = pd.Series(scores.mean(axis=0), index=unseen_perturbations)
    # rho = means.corr(degs, method="spearman")
    # fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
    #                      name=f"{model} (ρ = {rho:.2f})", marker_color=color), row=1, col=1)

    fig.add_trace(go.Bar(x=unseen_perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
                         name=f"{model}", marker_color=color), row=1, col=1)

    # log scale for energy loss
    fig.update_yaxes(type="log", row=1, col=1, tickvals=[0, 0.01, 0.1, 1, 5], ticktext=['0', '0.01', '0.1', '1', '5']) #tickvals=[0.01, 0.05, 0.1, 0.5, 1, 5], ticktext=['0.01', '0.05', '0.1', '0.5', '1', '5'])


# Bottom: perturbation magnitude (number of DEGs)
fig.add_trace(go.Scatter(x=unseen_perturbations, y=edistance, mode="lines+markers+text", text=edistance, textposition="top center",
                         line=dict(color="black"), name="E-Distance", showlegend=False), row=2, col=1)

# fig.update_layout(barmode="group", title="DE-F1 Score mean (↑) by Model and Perturbation vs. Number of DEGs",
#                   legend_title_text="Model (Spearman ρ with # DEGs)", font=dict(size=12))

# We have arrow up (↑) for DE-F1, DE-Precision, DE-Recall and arrow down (↓) for Energy and MSE. So we can use this to indicate the direction of the metric.

fig.update_layout(barmode="group", title="E-Distance (↓): Performance on Unseen Perturbations & Perturbation Magnitude",
                  legend_title_text="Model", font=dict(size=25))

# fig.update_layout(legend=dict(
#     yanchor="top",
#     y=0.99,
#     xanchor="left",
#     x=0.01
# ))

fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.0,
    xanchor="left",
    x=0.01,
    itemsizing="constant",
))

fig.update_yaxes(title_text="E-Distance (log)", row=1, col=1)
fig.update_yaxes(title_text="E-Distance", range=[0, 1], row=2, col=1) # * degs.max() * 1.5
fig.update_xaxes(title_text="Perturbation", row=2, col=1)

# Tilt x-axis labels for better readability
fig.update_xaxes(tickangle=25, row=2, col=1)

fig.show()

fig.update_layout(
    font=dict(
        size=25
    )
)
# Save the figure as a PNG file
fig.write_image("thesis_plots/results/unseen/edistance_by_model_and_perturbation_unseen.png", scale=5, width=1400, height=1000)


### MSE

In [53]:
# RF
rf_mse_by_pert_unseen = []
for idx, row in df_rf_unseen.iterrows():
    rf_mse_by_pert_unseen.append(ast.literal_eval(row["mse_test_loss_all"]))
rf_mse_by_pert_unseen = np.array(rf_mse_by_pert_unseen)

# Transformer
transformer_mse_by_pert_unseen = []
for idx, row in df_transformer_unseen.iterrows():
    transformer_mse_by_pert_unseen.append(ast.literal_eval(row["mse_test_loss_all"]))
transformer_mse_by_pert_unseen = np.array(transformer_mse_by_pert_unseen)

# TF
tf_mse_by_pert_unseen = []
for idx, row in df_tf_unseen.iterrows():
    tf_mse_by_pert_unseen.append(ast.literal_eval(row["mse_test_loss_all"]))
tf_mse_by_pert_unseen = np.array(tf_mse_by_pert_unseen)

# Mean Baseline
mean_baseline_mse_by_pert_unseen = []
for idx, row in df_mean_baseline_unseen.iterrows():
    mean_baseline_mse_by_pert_unseen.append(ast.literal_eval(row["MSE_all"]))
mean_baseline_mse_by_pert_unseen = np.array(mean_baseline_mse_by_pert_unseen)

# Mean Shift Baseline
mean_shift_baseline_mse_by_pert_unseen = []
for idx, row in df_mean_shift_baseline_unseen.iterrows():
    mean_shift_baseline_mse_by_pert_unseen.append(ast.literal_eval(row["MSE_all"]))
mean_shift_baseline_mse_by_pert_unseen = np.array(mean_shift_baseline_mse_by_pert_unseen)

In [54]:
# df_bar_des_mse_dict_unseen = {"Model": ["Transformer", "Rectified Flow", "TransformerFlow", "Mean Baseline", "Mean Shift Baseline"]}

# for i, pert in enumerate(unseen_perturbations):
#     df_bar_des_mse_dict_unseen[pert] = [transformer_mse_by_pert_unseen[:, i].mean().item(), rf_mse_by_pert_unseen[:, i].mean().item(), tf_mse_by_pert_unseen[:, i].mean().item(), mean_baseline_mse_by_pert_unseen[:, i].mean().item(), mean_shift_baseline_mse_by_pert_unseen[:, i].mean().item()]

# df_bar_des_mse_unseen = pd.DataFrame(df_bar_des_mse_dict_unseen)

# df_bar_des_mse_unseen_melted = df_bar_des_mse_unseen.melt(id_vars="Model", var_name="Perturbation", value_name="MSE Loss")
# fig = px.bar(df_bar_des_mse_unseen_melted, x="Perturbation", y="MSE Loss", color="Model", barmode="group", title="MSE Loss mean (↓) by Model and Perturbation")
# # error bars
# fig.update_traces(error_y=dict(type='data', array=[transformer_mse_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [rf_mse_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [tf_mse_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [mean_baseline_mse_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))] + [mean_shift_baseline_mse_by_pert_unseen[:, i].std() for i in range(len(unseen_perturbations))], visible=True))
# fig.show()

In [55]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

model_scores = {
    "Transformer": transformer_mse_by_pert_unseen,
    "Rectified Flow": rf_mse_by_pert_unseen,
    "TransformerFlow": tf_mse_by_pert_unseen,
    "Mean Baseline": mean_baseline_mse_by_pert_unseen,
    "Mean Shift Baseline": mean_shift_baseline_mse_by_pert_unseen,
}
colors = px.colors.qualitative.Plotly
mse = pd.Series([mse_loss_by_pert_unseen[p] for p in unseen_perturbations], index=unseen_perturbations)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.75, 0.25]) # 72 28

# Top: DE-F1 bars per model (legend shows Spearman rho between num DEGs and mean DE-F1)
for color, (model, scores) in zip(colors, model_scores.items()):
    means = pd.Series(scores.mean(axis=0), index=unseen_perturbations)
    # rho = means.corr(degs, method="spearman")
    # fig.add_trace(go.Bar(x=perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
    #                      name=f"{model} (ρ = {rho:.2f})", marker_color=color), row=1, col=1)

    fig.add_trace(go.Bar(x=unseen_perturbations, y=means, error_y=dict(type="data", array=scores.std(axis=0), visible=True),
                         name=f"{model}", marker_color=color), row=1, col=1)

# Bottom: perturbation magnitude (number of DEGs)
fig.add_trace(go.Scatter(x=unseen_perturbations, y=mse, mode="lines+markers+text", text=mse, textposition="top center",
                         line=dict(color="black"), name="MSE", showlegend=False), row=2, col=1)

# fig.update_layout(barmode="group", title="DE-F1 Score mean (↑) by Model and Perturbation vs. Number of DEGs",
#                   legend_title_text="Model (Spearman ρ with # DEGs)", font=dict(size=12))

# Arrow down (↓) for MSE to indicate that lower is better.

fig.update_layout(barmode="group", title="MSE (↓): Performance on Unseen Perturbation & Perturbation Magnitude",
                  legend_title_text="Model", font=dict(size=25))

# fig.update_layout(legend=dict(
#     yanchor="top",
#     y=0.99,
#     xanchor="left",
#     x=0.01
# ))

fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.0,
    xanchor="left",
    x=0.01,
    itemsizing="constant",
))

fig.update_yaxes(title_text="MSE", row=1, col=1)
fig.update_yaxes(title_text="MSE", range=[0, 0.3], row=2, col=1) # * degs.max() * 1.5
fig.update_xaxes(title_text="Perturbation", row=2, col=1)

# Tilt x-axis labels for better readability
fig.update_xaxes(tickangle=25, row=2, col=1)

fig.show()

fig.update_layout(
    font=dict(
        size=25
    )
)
# Save the figure as a PNG file
fig.write_image("thesis_plots/results/unseen/mse_by_model_and_perturbation_unseen.png", scale=5, width=1400, height=1000)
